# CFPB Seed v05.2 — blind semantic judge v01 calibration

This notebook calls `qwen/qwen3.5-35b-a3b` through OpenRouter in
non-thinking mode and pins the DeepInfra Zero Data Retention endpoint
to judge the immutable 20-row pipeline smoke. It is attempt03; the
failed thinking-mode attempt01 and unroutable Parasail attempt02
remain untouched as lineage.

The judge sees only labels, the post-redaction grounding excerpt, and
the generated dialogue. Candidate findings and the GPT-5.6-SOL
development adjudication are not loaded until the later scoring cell.

All outputs remain pipeline-smoke-only, `privacy_verified=false`, and
`benchmark_eligible=false`. Development metrics are not human accuracy.


## 0. Mount Drive and bind the immutable development inputs


In [ ]:
from pathlib import Path
import json, os, sys, time

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    DRIVE_MOUNT = Path("/content/drive")
    MY_DRIVE = DRIVE_MOUNT / "MyDrive"
    if MY_DRIVE.is_dir():
        print(f"Reusing mounted Google Drive: {MY_DRIVE}")
    else:
        mount_error = None
        for mount_attempt in range(1, 3):
            try:
                drive.mount(str(DRIVE_MOUNT), timeout_ms=180000)
                mount_error = None
                break
            except Exception as exc:
                mount_error = exc
                print(
                    f"Google Drive mount attempt {mount_attempt}/2 failed: "
                    f"{type(exc).__name__}: {exc}"
                )
                if mount_attempt < 2:
                    time.sleep(3)
        if mount_error is not None or not MY_DRIVE.is_dir():
            raise RuntimeError(
                "Google Drive authentication did not reach the Colab runtime. "
                "No project file was accessed. In VS Code, disconnect the Colab "
                "runtime, reconnect it, confirm the browser is signed into the "
                "Google account that owns MyDrive/FinDisputeEval, and rerun this "
                "cell. If Drive is already mounted in another notebook, close that "
                "session first."
            ) from mount_error
    PROJECT_ROOT = MY_DRIVE / "FinDisputeEval"
    if not PROJECT_ROOT.is_dir():
        raise FileNotFoundError(
            f"Drive mounted, but the project directory is missing: {PROJECT_ROOT}"
        )
else:
    here = Path.cwd().resolve()
    PROJECT_ROOT = next(
        (p for p in (here, *here.parents) if (p / "WORK_PROGRESS.md").exists()),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Open the FinDisputeEval repository in VS Code")

os.environ["FINDISPUTEEVAL_PROJECT_ROOT"] = str(PROJECT_ROOT)
SMOKE_ROOT = PROJECT_ROOT / "outputs/generation/smoke_only/cfpb_seed_v052_pipeline_override"
SOURCE_RUN = SMOKE_ROOT / "run_20260722T135306Z"
REVALIDATION_ROOT = SMOKE_ROOT / "revalidations/validator_v02/source_run_20260722T135306Z"
JUDGE_ROOT = REVALIDATION_ROOT / "judges/openrouter_qwen3_5_35b_a3b_v01_attempt03"
# The new root preserves the NVIDIA/Mistral attempt plus both prior
# OpenRouter attempts as immutable lineage.
CONTRACT_ROOT = JUDGE_ROOT / "contract_smoke_2"
CALIBRATION_ROOT = JUDGE_ROOT / "calibration_20"
SNAPSHOT_ROOT = CALIBRATION_ROOT / "source_snapshot"
ENDPOINT_PREFLIGHT = JUDGE_ROOT / "zdr_endpoint_preflight.json"
RAW = SOURCE_RUN / "raw/data_designer/dataset/parquet-files/batch_00000.parquet"
PREPARED = SOURCE_RUN / "prepared_inputs/nemo_seed_v052_pipeline_smoke_20.jsonl"
INPUT_MANIFEST = SOURCE_RUN / "prepared_inputs/pipeline_smoke_input_manifest.json"
PRIVACY_DISPOSITION = (
    PROJECT_ROOT / "dataset/curated/annotations/cfpb_seed_v05_audit"
    / "run_20260713T145423Z/privacy_qa/full_v052_v02"
    / "pipeline_privacy_disposition_v01.json"
)
ORACLE = REVALIDATION_ROOT / "oracle/gpt56sol_adjudication_20_v01.csv"
FROZEN_SOURCE_RUN = SMOKE_ROOT / "frozen_manifests/run_20260722T135306Z_files_v01.json"
for path in (RAW, PREPARED, INPUT_MANIFEST, PRIVACY_DISPOSITION, ORACLE, FROZEN_SOURCE_RUN):
    if not path.is_file():
        raise FileNotFoundError(path)
print({"project_root": str(PROJECT_ROOT), "source_run": str(SOURCE_RUN)})


## 1. Install the small judge/evaluation environment


In [ ]:
import importlib.metadata
import subprocess

REQUIRED = [
    "openai>=1.109,<3",
    "pydantic>=2.10,<3",
    "pandas>=2.2,<3",
    "pyarrow>=18,<23",
    "scikit-learn>=1.5,<2",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *REQUIRED])
print({name: importlib.metadata.version(name) for name in (
    "openai", "pydantic", "pandas", "pyarrow", "scikit-learn"
)})


## 2. Materialize and hash-check the embedded judge source snapshot


In [ ]:
import base64, gzip, hashlib, importlib

embedded = json.loads('{"cfpb_v052_pipeline_smoke_v02_common.py": {"payload": "H4sIAAAAAAAC/+1ZW2/cxhV+318xJfpAJitCNpIg3WKLOLYEJIgviNy+qCoxSw53x+LNnKGktar/3u/MDK/LlRyh6FMFw7vknDn38805s57nXex4LZIly+Sm5vX+JJW10iwuC13zWCuWljV7ff7hZ3YhRMJuTr8PXzKVl9eC3fBMJlxj/eb0ZbhYfNwJlpdJkwmWCLATNdci24NzLsEoEVrUuSyk0jJmWtxplnMd72SxZbpkO7ndnVS1iKWSZbGoannD4z3jRcJkocW2lnrP4p2Ir1XIoEzOC+KTyiIBB8VgBasFGCgB8oRx1SooksWHfWLJwb+sE8VUU1WZBNkGItiuATes3Uhxy2APZ0pU3KkvwKYhLuxTk2xFuPA8b7FI6zJnUZQ2uqlFFDGZV2WtoW5Raq5hgVos3LsdVzu4o338pGCe2V5xTQvt3g94tAt6X5FX3PtXxX7JfpPwHs86pkWTV3sysqjaVxV8hRf4VyVOQGu2o/iZK/G2RHCW7HVZpHL7RsZ6yc6lyJABKX1EXVQXi8XF2dtX7z7+8jr6/ezVxft30ev3b84u2JqB+RdRKKH9BcPfvfmf/jze6F1Zyy/GBZGCK0QU73ixFYm37MnijMs8gpmzqykSr+GZ2d6oOQoKOkKpIwRbSS1vRCQLZGpu5W4bmFDEYrxFNWkqY4n0iLZ12Zi8GVJkfCOyfilCDaQZHDSiEVsoVtZRjXTV0I0MGRI0BaUWnC2SqKpRDVCyKsFm/whVLJIGwZ3Ve0iqYlHwWpYRaonLzJE9LALECpooxS50DY1NjP0u2sHK0CFv6RXVqMiY3nGNlP8kqMhzqVQlsgxJjvRvCnFX4T2eUBKJgEY2OVRocp+YGSbGRXKLjOjTyUdh13ztIRobmXi9ameO1QXy1B/o6bRTZVPHYtUm+qU3iBHztqIwcJLQgwkUfUGtCe/KbCc8WTGlayhj8tkH1ESZKLZ6t34RWBGa1yACnLB/s3dlITraRKS8yfSaXi7ZVqxP7Q5RJF9N34XAQdOvQIscyTZjK7z4Htws7MDhxpknBl6APA7ZEgeFBoE56zzA6vK2j4MCKkcyecr0WnDAjloBjJW+BO3V1JaIqq6s92uicNa7iLldwwB+zfbWgEFMeRyLSlPobObZbwS7LozGB9G1JL93u4yfTOCzfET3tN0orrzSVLBUWziVChGpHX/5/Q/jnYBiCCvWtfevy9OTv/CT9Or+h+8e/uwFg2wveC7sti4d6MM5mKxALBodGxobnp8moOp7LhKO8U8mZ3IB1Eyc09Lu3IosLaosEcqPM7WciWPATv7WP6064GiK66K8LaCjMuDhE1y77djDZrE96LbLtOXQszR2cgkI+QeORHFW1zAo9f7uJHWZa6Uwo/aK3Ts+D17PvRY4Nos5zfo6eo/+IxO/l7fzBfSWU+sgTkxLwDdoOniCrJCxOQKoSP7KKmEAhB5sgwDAYKbkiPUziohi7I6lQX46MX0yA6aoQLoFW00OU6OZuqCtj9UGMZtwQc8VPauqY7QJNr2ezaI/I3uIsK5zWnZgdYyAFzzbK6kigvDJmvNhPYjIeKUvMrt0pNLmXEVOnXXA/+vxaD22boBjRK0jVwUxgDVDSw/qLDVGExauhgrTSjgqGvan9aBGnrTEQgCVL7s3zFyhPjCJFl7LLGtL3Bt5isrc0B/UHJ223eog0ebe2zR7vo4Zx6zSThU4iG7QvFFz10Pcuemjz2UmZjCOpgN7khiIkl9EtNlriiA1JG2R9r3KM441wsU/3N9Ypd/yQqZCzXU3uVuKbkQ9wbmb0xcWzOiLtbKpq1IN+z6Z540mQI8qWbkTm+bNDkoMB3/Q2T+5wdAGgzYzqpuihZfp67HfYxSCHh7r9DJFxNqa72N41a2hOJpCHwTKHSMaEyKGmycD2kEjXE/1ZwMckQDfKkkToyk9EFnfJ3ILz4OZGzpDu8m3HG+l3pm8CkvUDXqRjRfQvIgBK8kGtUsd5yYr42toxigufsbzTcJXjhIVwhP/xenL79g3jD6CJdt4XjAuFqtL2FSEnb7h156jBm3c+k7c2W9+4AylCTlSPBU+jd7wASZgYyY+rQjUOMq5AKggvJZqCX6xHqjgpNzDN/612AerAV/YlAfGTKwsycScTDWMQnpSfvBwXFJRhRi165rvD+VdzkkZC9AlZY4fXB2X4BPFkukGGBs8R8jVo9qb81nGh4x7D7iUmeVQJeFHmSNkPK+O8lClHcj9UdDNootzVvIkcjcy04w2tUURpWN0SZG/coepGePvkOImke1TmJXA8V7llgZIE+Ii53MjtPeYEwGE1ocE43AhDKQUj9xeo1yAuEWkkY8LDsx1a8+p7vWBdHLB4d4Lib0ZU5Es9N176FWYVCJQqqRjbO01Oj35cb4qp6qH5D/lE+BZ5ekbybYbSR96E8KBsmqzbeYE+6WoGtzFNcCNjWC/ovn4jY7DD9Z2NAyk54PXVuctbuJEZIwbBG3pDpO5uE26A2M2PAsfhvk1LnJ8+6DWH2vKLnEHHlF5bR7ngesWbp36DIktbsngtffP4jiuuQgbbUeuddBmzPONfxPcsSl/nCUkV9GVH1exlOtznimoTI1TBCixJgTsW2Z0cB6Decp6LDKFP0G1SbM4W3Gm9z7MYMK2pwDAPFHDZE54kPmH7FOkkg7MZStSX6qC25U56OnvW6iXhQKWsk2zxaAHM5cyRxgYpyRgMUhloh8xGGhq6Q89Mc1n76y9u+Iml80Gb4RBR9xmJVwNwckOilGsbkap/g01/p8bXEF2bfCKbcoyM/Hss78bYV1o0xo3CIRcDl5avji76C50bTZdC1FF7exV8DbFJul+ouR21MXNSYWoe3uMmcw3cEWWGj2Ow9kwZPTqcn6UujIYPE7tCVjR7rAqq/lhzOwjgBx078GB7Plx7Q8Jn2XxhHTAxLp3ZWhvgNpJ0LfcR0PHQUaM1QGPIwPUmAvo3AQxD1MzAP4GXQLdewh3sdHfZ9wP2I3mPbC9HKyRN/E4LBIiGZZCO01Gn9y95sGR3Wfg9ArUpX+3c/UY7bGkPegWBnnacsbeKb+viFy7e+j4Xtev9343cHfq9IGYyhhGo5N1OaUiZ7TvhsHpdrgIIVh25pjGpJ/t/lvNU/tj0x79LxLIEZhfnT4vplgvSaPPoWslztvJJQhxw8IRDh7id6yIUi34X7dQqsn9F1/RNs26BcB92E/istCfInswOnjMbfF/AGZVvsh0HQAA", "sha256": "a35e8fdfcaa7e486f7b28d67d587f4cca8ff76fa5c52eb6ba8806f44804d708e"}, "cfpb_v052_semantic_judge_v01.md": {"payload": "H4sIAAAAAAAC/5VYTW8byRG981c0lCslbQLk4iAHraUNHGS9xkZ7CAyDbM4UybZmumf7gzL347/nver5oLw+eAHDJGe6q6tevXpVrb+Y19+9+9acvvn7zd9Mkt767BrzsbQHwcO/rlb/C8XYKMZ643wrg+A/n5elPxfbuXwet+xDNNaks89H4du989Y3znbXrUtDybJq8SMcityYh5Ptis1igu/OBhtMKsPQOWlNZ3fSpbU5xFB86/zByKdG4pDX8KM1B/ESsbO9sPaI/U3oh846uOcSHbZdJwebXfBr40M2J4lur/bxuDM5lny8WULcdYjQ5MB9psFBrqV7iIEuwB35NEiDY00rjUswi2ch2qaT2WPsWgXEEisg6cbcBz3b+T0e5qPNACiGZ5OOoXStGWxKMGL21nVmH0NvXE7GEWT6GuHfj5JL9DjdNhlIBS/m3//94a0Ju49wxzy7fCR8ScyTnNOr1Wq73X5Mwa9+XRlzNfl69cpc2aaRIZvfTBTdyi8nJ89Xa66MYrErYeH7q18kBrrVB+a+68Izwq4LAHMr6eqD7sF2uNoIN+G3Mb9epVCiPrha0vfbkjN8V7DwCVzkam2usnzKXA9IYq5xgleB1Ig4DQRCbLr4d5zxYfU7I1yt7sfITCydMO5rs60Rbl9hudIJBnHkzNYawT/Mdox1a/qSstmJkX7I5xuaqNjABFLVYVlWxP+UMR/89aVBYgyDLzg+IUeuApzivCC5zqey37vGschYTP3OHUooqZYWwo/SObsD5Wqot2MqpzTPhCvAbDra9BJpfSeN5XM60tjEs2GxdTyxdJne/pQuCvJLiYcRPLzg9Ymuenmur2njXZRKdoTbswiBykVYo7UkebbSFoDSVDEQxLJnSOCORPiXBIWLeq3HFw97yYhtjvqE+F0oEzyfU0Vf/qOFqfUNJrWOakBG9pItyttWZahMHRXBvIaOcAMC6OfygiMLn1+okJlVCOf9i09BjlQOKH+eligpODuT1sQEOc4uF77Db9g8HKlWQQFinEfrQYa1JjtDz/DRhNhSXtKTeaaGQE5C7FXbCIAXaaWlGmHtz4WAN52l2jV1DcKBuUoZZc8II/w5w2oSFB58KH7h+RBDgyVrMwTk5rw20R2OUOBQMoRW6B6Oa8W20E2NfUzmyMiRc+TtIdq+t3ENJLyszY60zGd192hjr54AcTscI1gJybsbifdZrU1FjiyF6H7R0DYpIwkbgnaQFlW25IWgk3D1XQJyotKs9Ie10iu/JnME0HkwhJvXCvrB18dwFP7BETeoZefVSI7WJ6SVdUfHgLnrN/k8fJ0/1Q8QbWyOhjvXoA6obZMyP55cI9dTk0UdJEpBtAVO7UWmJzvbWWqJbT9Cgnocoj5Th1roBVadYW5cXPwS8R9C2OM7TlNUS/qqMKwBEUF6ULfu0g9F6bJo5sAApWtvi+cH2CX69labtyjyoGnhUbeYHsavyjYsvW26kGamp9CdhKamrxoC+zWSlzdJfEI6T7K5KJfNoaCvAywqfEoOviIWKDHalmU4qtBnVvkuBtvWskEMgLS2ZTKXCPJ7M44XcKe2/fnMFyVKEUHdwJa3Ed2f1l3WWcMku5e5+K/H4p/K+IWc56Mb+0BLWmA7DE7KogPaVwnL5A186enI7DKsXTg9YrkI92ZOJekwa6GjwKHbokCS1Ly0VsekPPdg7Vy9WErvvnTmcm7DfvZ4ib5WxIXlnKTbV4Ue1YTtwbbQOjJrEWDKZKcO62yxeLqB6T3ki/2ce6eR5rNRc2Ywjph24IRJ+y9JTOkfBUSXGbtnt2ITgomBiq0JoOxVBycsderchLhRIUVtUS1esLAKMSuqTqjKtk5qPUMJ0bBCPNeXbNyFVIxyKF19QRy7kiZGoo+qnoGMeqQCCVKhAbh0RPC78xKZunih/xvof1tQRVX+X/ip9Z0+oxzKI6UiUHkv+TnEp7XmPJJ1VY5GizqvYZru3MHtnApbEx3xpxGq2tRxxn5FZdM9teXQh9ZGdigKVyN/Oq5GWhR792UpQApREZqFhHmfPXQS6LVWyIl9fbpXSGqgvfV7FB6hLMmu1882NIXpW2QiIp2gP3/D2thkqYr2BA5rbx67OESsZc/yFyXRBuD+IlxUyc00dejEoLpRdeLlDECN+lxmZvWII4qeqvEHzFIjEC4XNrXSvtgO6iT4oh3AcojrGiYzD7hOyuWLrsN3A1iPdDfQLB8gDOcFuPU8DmEWrioc2LQbF7EgVXLYHTNWr09L4s33ztcLBBufYqBTJKw1WhSznzoGL8MRBpCHaTpf7hbfwda2Xm7+uSjhenk2IbLVPE+PVWDwiAJuzcUVh/297FKOzGsTBqqRRqCXxFmgkszNmbP5ZJbXoW2d1Oule7onNTq8uqQ9AIZwkWVicSsBPSu1a4VQkhPOBuH0snVj3nLQRh6hIZirjw4668fJiz7qMV3Q2zju2X5WtgfOF9PYN07301VomSd55OWNarkAZekB+bvoTrY5q8Am4CGK+iPsZbN9/+PD/d3rx4f7D0T8/bs3bzav797ev7m/e3zgI27avr/7/oef3j5+2LKcnLKy5hzeNnIMHe4T6cWgyikQQz+nVYgXgwQVk+7h5YBrtUHh3+AG4aB7nfrwxDvKCfeggRRdImE6aruq+l37NGacGhppi/gjOasc2WHXEYPxk7EDygB/ELlZ/R9XhByskhEAAA==", "sha256": "54201b7ff4ed7d1a947657e0b09aa49ff768af759d1b9e27d92a14eb2cb0b536"}, "evaluate_cfpb_v052_semantic_judge_v01.py": {"payload": "H4sIAAAAAAAC/7VY308jNxB+z1/hbl82VbLhQNAqUipxkDuBICAu3EPRyXJ2J8HH7nprewO04n/v2N4fzoZwd62aB5TY45nxNzPfjAmCYLpmack0kEXK84R8LZMVDEWePhOGP2ORLXgOyRCleMK0kCSBNaSiyCDXhK0kgPkWBUHQ6y2lyAily1KXEiglPCuERKk8F5ppLnLV69VrclUwqaD+/VWJvP4ulNOE9kDzDGo99e8BMX//Ejk4uYLp+5QvarFr/Ok29HPB81W9fpw/93paPo97BD9WQMWSF1pFK8hBWg+jHDJB0RKjCSi+wvUoXhYLut473KcFLwBhAqoy8QC4tk8RoUzktY3Q6jafVLCECsniFGis1oPNDQUZyzWPqcHbAKhaAXXP9g+P6JKn4Bb7P+pwFSw0/Ibn2y5DlQqUrRjPla7cr52ApxgKTS5FUqYwE/qDKPNkKqWQY0J+JqdcQqwJPEFcGscI5gpkC0gSSMiJSNmCyDI3kYva+/wItMYIRhTGBG8pJNzlYighgeWX/xn0H4PzO9z8BtC9Hr2+uTqfnszpzdXVnE5nn8kEqyKCfM0lhnwFOgw+nM1Ozz5d386n08/HFxsngn7P/4mHXZBNZYRbuvuRBCXSNYTuznxJtmTsBqQKKh0WKEq9oxGWs4H07uBLr9/bNOspswsjEohSF6VWozaRRw5KQz0jC7QCSDpoizVIyRMIajUSquAYchk1JGUCMlKilDFQTDq6v7d/tPfr/v783cHhwd7RHwG6eHJ8cfb+5nh+djXbRMn+QN2WCtVIFJBLdBck/fMR8gN6SA8OF5QdmFR4R5nWkBV672AUo/WFuwsaNCZOpx+Oby/m9Pz29OPldDb/hDa2zKKl7cSMDB+mQaPh6ub45GKKx+sjLmNGq0IfHmEEKEvwLI9r68a1CCug1XBydfn+bDY93eFCnZLJqAWUSjApbcC0/nju3M4x8Xao8nHIQEseK+uOU9HrYSm0FWBBdrj/4vK/wYAaXh/bhHM7VWF3l+seVbu7dcym2sZynwx/JwiXvlNaDkxn+DLeNI5328EZ4aaDfc+1+lDLQKHn84BI+LNEljTEVqSgYTKXJTRFp7CqG9198tPErjgF/XFDHpJxLMPPiB9Y8g2Dc4OhbdeVG7GpE4ZrCV8uQQbORIEshFe2QDGMhbnj3y0BmmrjyXjDjaiS7JMlsnklMmhQIjxvEYs4FoKqOOSla9EU0A6D7ZL5BCw2bSYYk6Dm3SQYbIpI+Iqtxoi4b69JrDk8Ogn7rd1/uWtul0DMFaZpy8z/6po2hy1x4QV3cHvbZbfCMKjCNtjCa9Jd8DpTJ+vRsqmvyOSfCl8rCRPLhGp40iHksUhwMpoEpV4Ofwv6HZVV1aLOjiLXeaqcrqSCJoFxyiOIJ16a5TGEXXUDW3F9Mxd0t5xeKR5Rm0n8FPLvSfyTSg1pp9MKjZTFD4rUdVbXxabHDXBtAgY144E0eWGTcO+dlzxBUcpCKDA7dmCmaZo5EqPeYExf4UBfS4yxMFEtdYya6rk2ysVjWI+2Ee71I64EpmTGdNj3jlugxj5K3mabjY3lsZeivhd1FNq+2Z7Yil57juemdQdbldsSo5umUMIbq7q82anYKqlePemRaPdY9w5e23pV1WuV4el88UMtUh4/b19zV5wNuihtOL3j5H2JTYSuRJrg/geGY1RXAFLsGqXesV1Ivmbxs0lLvuSwS8sC6/o+Y/KB4rS04osUtgX9C2K9GolwU8t5+wKseQAn/7RMsJ1gLcmM51xhR/SKbomFgHSiItJUZLCptNaUIeXh25Doe8BTTyi4S2eRlsqKXVxcNuwbdfW+F/reljkOn6RiXPLxej48jI6Gn64uNl+r3ow0sGxlA0NMYKJWcZUOL93hoZpwo+wh4TKsxl3bwgeIEDdU/+B1dP/ko8Su4bi3MWPpOimzQoUuF1FLbsChTMWcT1zgsPckaGeyPyDK5OsDPDubXtJ26dx/wEjAt3hecV01fNmHN8UXOLYxMwjVb/FoxjJQBYvBka5dlEiQjcCxXJUGymu7E+KD075HEc8JpYmI8UXgnYxYkhgz9kgYDIcNAwQD+0Sa2GkMY7RkZaonW+Pym8ocKbytyY3Nb6pp/sPhIHpbXz1Ev+2YDfw3HLMD9EaIKm1+cKp4ZZjWLlIzbAwuNkYAI7MhvdHUXpuwt8friTkavfIe9ijXyfhP1F1Tt5Ps7Hg624qodNoFP12R6RDE/1oaBjjziqU5JjT+H2oyIQGlBkZKA4efw7T3DwHtsvABEwAA", "sha256": "cb2b1940a5643f4f44824a5041081e75ca20177b2f1c2fad67e3af45900b0715"}, "freeze_cfpb_v052_semantic_judge_v01.py": {"payload": "H4sIAAAAAAAC/61bWXPjuBF+169AuHkQU7rGjmc3SjFVXlveeOMrPiZVcTkoioRkrimSS5C2NVPz39ONgwAP0Z5M/DArAo1Go9Ho4wPWcZyTnLHPjBSPjAR+HC1zv2Ah4WzjJ0UUkN/KcM3Ikq3SnBE/2ZJHFofjtCwI36RPjEScsGc/LnHUxHGcwWCVpxtC6aosypxRSqJNluYFjE3Swi+iNOGDgW7L15mfc6a/f+Npon+nXHIKgXMRbZjmo79HBP/9nCZM0mV+8Qjia7Ir+JQdxTaLkrVuP0y2I3IWFSz3YyVrtg3lWhXJzz5n52nI4hE5SpNVtD6OgmJETiJY+WBQ5Nv5gMCfGMuDPMoKPlmzBDji6iYJ26QUpPRpyHi0hvZJsMqW9Hl2sEezKGNxlDAqtAdtezRIN5s00ZMPBW/8u1mcH17cnh7R68XhzeUFPbo8XtyMqm7+6O8dfKSrKGay0R2w14BlBQHZy5hdpMVJWibhIs/TfE7ID+Q4yllQEPbKghIlJWlO2GbJwhA2/CiN/SXJywS1OjEL/BbRcRLQNpsTWDbYy32SjnMWstXDdyxqMKBX15e/Lo5u6fXl5S1dXHwiHljHhCXPUQ76XrNi6JycXhyf3lzd3S4Wnw7PaiMcd2B/wmCpY7SQYYu3O8kZT+NnNnQFVbQiLRrRwWLOFA8hL6XW0AmYNUsKfr//MHAHN+eX/1joyWuyTIkDRykrCz41FjSVCk6TeDsV6ueMhY09SJ9ZnkchcwY3l3fXR8D97kLPYE0H/GFL6d5s7+Psx7292w/7B/uzj/92Bt2keJKjUB7Sqfqd5rjVU56WecBoN7ejw7PTn68Pb08vL+o61pyFF+HTNGNJDgtmOf39hSX79IDuHyypv48m9oH6RcE2WTHbn2pPBILAdA7o8Hhxcnh3dkt/vTv+BSReXF1e4zStmfVkQtKcoWFO0K04NQ7ni4vbmx3DteujyGeDuygYxIYDmtidGLSDReURp0ahShhUZkOg88Xt9enRLnFsVWxYkUcBR2U1eIBVnV91mVcgPFjNvNRmmJNdWzET3Deh4X15fXh0tgDelcnmfhCz6TorDj6CwVM/hIFRoLdLMAj4s+FwffgvNLaGpaLF+S/TmqsUX5wVUzhAv5esGOPR4tOlXwSPdIZ/E9Vjr31xdXi9OO6eIgO9w2EMaZSIcybcc8eJkodub9ba7Ivjq8vTCzHNydnpL3+vW7f48T9Y+OcwpywJszRKCgoyruJo/ahN1Zj75d0tODVL92oe8M2fWdK7h2YyxXUwCGKfc3KDRlSICDesYp0rg9oGf1NpNGiQVfwbstci9z0H8oBlFDpuxe1nDDInsEtDi6/ilqcxBANe5OILI7T54tFnRpfbgnEIF0kBk4kAO1wzbyZdrwwGYkTVm+Gi8sTLnf/cz8Z/8cerhy8f//z1j5ZAv6ICZFZz7ifRivGiQ7SN6qLgSDnY7VwnBfcO6M55gBnFD0GsdNqi7FO/YGGieT+tinVi1ZAkldyaJGTPLE4z9ERUbjsFIwshp6GYi1GwNqocDkjXnPYbR1uCKOqyCMym+VmWQ+gJYePq+7KJEhqzZF08eh9cy5ISf2NZgFxwFJqWJdgfLfPYtETJikHshFgjrXBOwLUU99A9wuRNphJ+HKcvIEbOfDBtoAzRjOKIC0JJY69cOc5OXks0YBHCNYvKpCVBlsZRsLXHLtM0ln2Q06r1DQaQ6AANz2J/S9HWh9LgMUdwyfhvSKWOhUwTQtAekjQyjirBlKSQRSfVCCCNYaOeGS3Soe3qDQ934nOapTx6VfxUVvgJ9pjJZLDJXQiBO6UWsYyScFid3RFprKNSz1xnSKAFySTiQpFD15rDjyBRQvJaRiqU4w4sKYwjMWNBBg//MRkiDvNaWnatFLLyK54QCc8TJGRw0rGnmWp6VsZpc3KVKlbCj8iTKuX608gyZRXTjYJMp8gcWj3mqLW6dHBvtsOJAy/eapZBuNUMEbWDgwqBLeIygZjbapZpfavZJIStVbUCWVtckea2mpsOpdZKpfbn4ryNBsL4Orx7zQwbQxtmaA6BMTLhJlUVnLPfS6iSoKZ9zeDQR01+3m1eQhm8AtcMtM8Re8HS0krR9C46FXu3QzpcLlgklI7tk2KJ6FjkWGgr6UT0bdog+BKM8pM49UM+bFkn+Ac/pAUE8SH41hT9v+eUxWr8k+O6Dcuss2pY7NuMOhLWOkfb0N9mB5qzVyPLvSqEBlgI+WsMFpssZgVzXNQU6hq3qk+5wpJqe6eZIYcoMRx3ywFmAiU1BqL0hcPUf/DI3gzr6jap9gk10rflWzUMU+A0SjJgMAZudqQD4KZQAi8hrcUQ0xZF9Mh8GIMs5HVB4YzIl6/VSgWFJMY9gwVKd2O0e+Jj/QsrbZMGPuQXCNOYCqox7u1lK6MOUyaHiXOgliQEB35pLHOWSmpjqv8/K/mknZ59QGI/eOJmGyqwTM+jRAICCKWgEXMMmiIqL669Rs1ZRDxKIHpBKjRschqJZMRF/Te7JN/3GtlRjylpqSF02K5HSdhxzL9x6rbXJJuSF1KLmOLhWbIFkpzfFKDSiAlYlX5RrKbG3iskmLWJgZXIkIv4aziku/dul5zyYCLMY+RTG/vmARnjsPfsljJEVc+1fYHssk6/HmuKwC+mmjBJvWMyKUcn8XabTvXtNu6DsWHlAacZEy67D0tVBCgAM641pxnNmg1PdgNWHjUKyEG4qB+gzvHjYmv3wcoBPJbBrN278V8hr35iCa/zS58Bacspj8u13SGKELqC/yzRH9TnEUqEEJeDriBbqPUKxCNI4xhiBzowqwtggTofrG9QZSDtMmbhjs7XIC7DmuIQ4WiuTQECLUmxDEbYF2tBzmDfQ03yVWamEeeY53iEg90wqA0aNjLGsDOUHyZsq2E9tryqe3vpVmXdqWy2lAdnTr4obl8tByAo7lv78PDuWHMip5J3HML34GkGPRO96cSwbc3bsckP7w0r7ZkVN+H31KZAhWjxbs3fNKMHdG5Q8CdbRwYGSYYm9R1ysQQgn4AR5DvGKcdmSuES/3187ehKU2VDoDmIh8V2p/OYEwfhsanAyCYHY8DIxoCR7T52c1wYg4ucVe53upk5aWT0PUhcxYkiCufUxyWQED1GyROeK5ko1bA0K7G3jwhkUKUf1xb+xKCkkZsgXC18uwR0SeAH5DBtbX3VG9xkBtvaIn7HJlYyq+2s+IXRCjAWLi94MKGUpQyYW5W6NXfU0kp9U1s+XW0t7CpNVyuooXAlEAvzMiioRKHjXY5/TmaTH1veH1t/aoWAOWQHrTCApLO+WDAnHyYH/QEBSWa9PnguPUuvK56Lo9ZhIrs0+W5zsRhY3I3Z2BPYlmO1f0NZ3LAk4RJKzupm80+54URvOGmfqXElk7YYoi2mUSrrqrWCEgCi6UxdJKAvUxfRYvajA5CQQI8NcILKOucCpdmo0A5wozdHW6gxpBrTPnS6fAjV8QQwRudrAoyENd9XUwhIDu7ylpDjUYkIwcItaMhCwDStwHbkvR3QWlhPi5YHj1DCGFoLAGrRmqS6Iq8DQ60RlSdvaxIG71Jvk4sN5wLIhStSUFcvqca+hLYsGKx3kKrKIGDSxxJqO7pOY+RgYW69403dO2pgga1hzctKNaIG3vTOZYH3oya82Duwoy4BDjY4o4ZLpHvlg1GGVQ7ZeSLVXbZ1JJ3GsDrM0Oi06scRmVkHTNGhTHjFiagvBJRh53AkguEO4EeuYYAHagJwGljbsGGVdS7Uh6tM3AVrTsVIXxmBCF13TGay6uLE049VJkn6MtTvVSbQ4wJUnoJ33yA4bXbJAvy8DqzQEJpcykNN6FzPSrEeLGqdJdVoq9TJptSlXI2yqu9syuY9jWcnPPfw+8GOXt2Fg/vVWnrHlY6nhnW+GrFk6bjq8b7UQ5AptiE0d5j+fVc5/jCqM2kjC8isic1UY6zVWZdMnvjXutEQ10tNeXedVLUCk1xUI5rXeV2JiiQ0Lq2bIMujZz/Y4oVntIp28lnC7j9u/PyJwg36OlqK5K+LUNh5DFftMThUtc9tUktbeKvmNVKR28eIKzRUoqC1l2nVqzQLYCPHHfBIIzPZgLWVWHUhhCMIXyJwMr9c3Y4PJh/HN5dnxH7dMBJFlFAgpFsBVKfBdtLkeZxKrD/GWwIZokfywI6sem5EZNTlI4Gion0QNHpxanB9Tnt7zbu7v4q3eGIKhV3DuYE7L+KThL0o3agL60mrZnGbVzLqvdJk8xRGeDUnHi+Jyw6IKa9wJ0rTJ/HZHvmSw0W1RPCNv0GwPyw3GYD9yj1OpGvCxiH+9BzxKMHFBAB3gPo8iCJP2gR4DKxVvD3rjDevB+yVqBtEPZe6uxPmu1W37NJyKnFo406zfi88f4e/VwtSngCwbliO0UBtnr5LDmsVGi8x99iYAsroGzz6yZqFnX1oMLAHG/SzlbrtW+1Kqqq8az8UQgYTFNY4eIlkVmO6Lnfx7weyqD8ilHmhSmIlHBmkOfbDNRbGEoxPE/JJeBd5rhoMwb44xGY4Bn6h3ySiIDrTlChJhuYNKIlIs9Pf8G2jjPM1brAMZ+pIUDYxy6yvQYGkRZSUrNah9kSnDtVwd9A7VMDTWm/1G2gsLwQbc1GNDsCuOKqhriGWb2Hqs0qb2CEbVL5lXNSLdcuhoz6GGthDtEgyc1vYinIiMKSyrVqHjTmKpys2pWypleliRqSRv6w+JQKGVPlrVK9vUWS5qnuzkod3wIvqDu3ZMjiV3gGuKDlqWNG8t4BG5UfEE2UKb5X5UPgK/Wp5cgF2zDM/UIiaaMQnMRXBYb4uMahciZ4hZDTiuTDM71EapgG8GbVGTvwwxGnEkKEzHgs9j6u6AJ/WeuLqHBKelQ8Cel2vIt9kqSuTN/iJN5K9zGqlRw83816yl50pQnp4qceSvYyq+riHj3ww2ctGXXn2spFvI3vZyGK1hwe8jnxjOVUJ27sg+QpyNytz0kAogQZYOJzhbGXVcobGI2dMkESJITAGat61SbS04zFptnVqicdbwkn44XuFe8fL9W+WzOT93ylclTb0SfnN4mksZWwBLO+Rc/db17dmx2MiksE3jol4xtrLRpe74+UWsR1VKFpJZ/+4sXTyMNQXVxMeRCMoDCjgzhrM3sFApoljkeZiaOrhUL2XE4zsuKBCxcaPEhkkLqDQl2EBCcRLP4taI7bwNVFpqp7fxDMow0BCK5fuSmgFC7kFb+XSbuPdXzPJbb9163zo5okp7eY60GDALkMp2qxcvg5TSTrTaMEcFholqVpFtoWBSgpVcVUEFmgnCWTDyMobXqxe+LKZW5ChZi+brPEGWVUsRIMhsOBUSSAbDEEdQJU0VZtdAnVCpUp9rU5LBaZe8yyD2YE7IYHV0KLSb+FsQtVWcxdN8/3OUhDPGP7fOMKDwv9X5sFlJ6V44ih1dMWGx2/wXz8hE9LWNgAA", "sha256": "651549e7f7c1f130ba9729f3fb5e21d0cd82b84afd3bd9b88bda831b3410ce72"}, "prepare_cfpb_seed_v052_pipeline_smoke_v02.py": {"payload": "H4sIAAAAAAAC/7U87XLbSHL/+RQ4XF0FsEiIku3bLG+5jmJLjnJeWbHlreQoFgokhhJOIMAFQMlararyGnm9PEm6e74HICn5sq4tLQHM9PT09PT3jO/75xVbJRXzEi9lDauWWZHVTTb33p6c/6v3mbHUux2+jg69VbZieVawQVnk9169LG+YVyfLVc6iXu/iGh7KdTVnXsVyltTMu05qLyk89nWVZ/Os8dZFxW4zdgcAV1V2m8zvvTSrV2WdNVlZRJ53cZ3VvRXHpvKaa1axRVkhwMW6ZrUHD8sk9+YAvkqKOet7t6zKFhl8gsYmMBg39bKm7sGAKaOmFVuWt6JlWWVXWQGgiqSqkia7ZYDlnFWrpk89a1aktUfTTHopW7CiZoOsGKRs1VwDpDSZNzCLq6pcF2lWXMneXlMS/CtWMIQLiKyqElGoop7v+73eoiqXXhwv1s26YnHsZctVWTUwaFE21KHu9eS76goIUTP5DOS8zrOZfPx7XRbyd1nLXxXjQ8zLPGdzAijHeAvIwvL2YZUXyTpv0mze8MZp0rAmWzLZUj73Pfz7a1kIoKukQQxks3N45B+a+xVSQbw/Ku7VJFZATmSD2lulvV4vPv/08d+P317Enz5+vIiPz372xoB8xIrbrAIWuGJN4J+cnr07/Xz+5eL4+OejD1YPP+yZj9A56HnwDzEJWrDDqGJ1md+yIKRW2cJrtaEPLAdu5TDiRZbDuhhdI+TGoqknL6e9sPf5+PidHNtCZd/zgWzA9s3+fA1rz9L9GrZOvCrLvN6fL1azGPlz3TD+HrbUoQ+zOf356O1/ObMx4NKLDtgGw3DgEmicrNOs8WW/al3Eh8PDPw+/O3h5cfDq9avDl3/bF7sv/iXZX6zznHCBP4APTPCnj3893jTDct3ADOp9zd/7JAZi3Co2HoexFBcxbLuqgj3g994dnxx9+XARExVPz2CNYRBNUhjBgaEHirMCho6Q63MN6OTLhw8EbTscXMNf1qxxMPjp6Oz05PhzCwk9/jIpsgWr+bi6N/Lnx8+nF6cfz4hKxiJCdzVxSWdDMAHYAwfYxy8XQAiA+Qnx0OTH1WNzgIVLsy/EYsrpUIMo+aOHMndWNg3swIM/eeWCZM/L/nD4yoPxmgEXVCiFhHyqQS4UTZKBUEgabwltvINDAFSUxWCVJ3N2XeYpil5YU2gTlLOaVbcg6ZCFBsPvBoeHIcjpT+Vd7c1YXt7BiABohmIwqUBYgvj5Iw6Wr1PoNePim3QEyIe/EH66MQhaoExTZbM1l9kk6udlxfsC+EgR6afTs/j9p49fQDScvY8vgEhnn4FcgP35h6O3x//28cO740/x+dHFxfEnXJOKRfNyuYLNHFT+5SR4Mzo/PY3fHkH/d0cXx799On539Pbi+N1vk6PB3+LpXng5BeFCcDdBwZbJ4Nfh4PvpHsCb/NP//vf/DKbm2/AFwOidn54ffzg9O45hAM4gmiP4bklymFus1aHf+3T8H19OAaf45MPRe5zYA21gn7RHDcRJ8nFTrZnf5+9noNOul0l1EwPcq2yWs/EiASkmv0vOE/oxVV8fYaj3x/8Z8/kDejjYhDrZJJvhxGBWUfynvcF071/kI/y+jPBh+nDYf7yc+ahZo9Ow34YRvPnhD5dpCLS63HtzMBlEl/X0TfgGn4M3l+nDy8fL8I18Tc/iAX6/egzeYGd/C2DqMoC/h/R3a5fL2XXTrOo3o/39y897v13O7u7uLiP4uRt/AP1934A9BT0GKtSrr5PD138mhRGgZhyRCgm9wY8eMPWI4KXZFUgPILFQ3hHvJBTSXQbWBHaNyhUrAr+a+SGqymvYGjnjEPAf7CJvlpfzGy8rwKphVZAny1majERLUFZJGhwMD195Lzz8X9j3Zr4faggal2i9QvUeELxQTBqMkUJ+v2Zf+S9A0ppow742wW2Sr9kIJ2hPVMBwpkmtQbvPy5QF/rpZDP7ZD8OOIfIySWOS7S4pc5AQEzRVJjBWH22L6XTURT0aBaTMWI7TRUmB5gSHinDQOsB9GRKJ8RdSmHdCcwHfRCihVkEol/2ughVo4QpcBEJx1IkuzeMMrCiOB6HMjYpoeZNmVSAsjPEF7PA+iE+AEZc39NjNKHfAte58+17B7hDhsX9ZbOYjQBPnSNha7CFYiaYXEH3S9XJVB9ASB6vRXk3qeZaNT1CW9MHar5r4ht1zvENvz6OBBZmE/RRLAzxG9AFY2UiCFcmSc1IfeXo58myyEdWwpWSwnBvqY+wSYA9uLiJcn0ML0WwjPRb4l5fw0t/3Q0VztK9wfFCsEliHhYgt9FvQjA2Si1a6770w1q42tleVZGBB/ozcflxVZRUs/HeGMyJp4LF6nqzABwHPxLsrq5t6BbiOvAeJz6Nv7UgcTZCTBPm9tqkMkyIQkkbbGAZb0jeyZ6QpY30lIjdrcOEcpu2DvR69A6PzpALCTkfuGEBMYw+5g5NA4gKjtS35DMEpZORDSVXH1ZatJ9GqBHh1DSpy5NHu0G1hPuskj7kOjVesQu+QGnL+1C2FPxorv1HDdJu29WpHI+6GKutufs3mN3EN1vi6htb+PC8BeIx7FuxloYLnZMQK9fxIf5dZvUwa6G0TAXbUyDMpSmwOb0GokzwNrf0MH8R75FNJ1gj3Rx3opsDZXSC9P4x5ZwMvaKpR28rj52K5LMZYleDt3ysQwN0amuRvGAKcmdYkYaV+WWdo5HJHI17kyVUN+3gyDQlV20zagls3ahK8R3DJ3gTcanRdgZLrQlIPhRhJDbHCJDXGrS0mBAHnZ7nFxy06+/IbTOThUW5wMVGjn7kPQABlaVbGKCDuwcSgKEMtmR38gWhe3/oGTxbsiiRIjAZ+VeZbms6vwf5kxRWDNjXEAequRrNkfrNeKaFhMa5YPYk6rUxrQs9eHEWJrLgFqpbgIWQgKQu0x3KIS7WWyCQ7rUdtqhCUb1OkKscZtwqqCK5raKuIrnKraISFsujWYCZP9A2Q1lZDrUEcQrrcAq4pcgKG41nZnKA/xAmDXSw4rolJpNaqj3/2O6Hv3qiK4Gi0mfsVJyV3apvIE/yMlOXaiYgr+BNfplz2AzsFmwA8lb2nmh9TiDCxMS6sISfZKhahrLhIxo6UdvTOoM6uBBPzickN82Skd+2w3xVbMk8T8In73qIinpMbIUZLjkwUBSzwJUVB4siffe/719+FfaORnJAnJlSTCSmo0geXfyiaG/yFNjEYoIQDMaOFxk4+/KL2r/dA03kke3SOkUlgPA3aZD+xn+jDRKr9VZbFyEYgKfxpxH6B6ZR+GIFkC3ZvBzl2WhJk6OTNIRB8T/EJMA3WYEfAF3O3bMRHGhc80MJxcSIAz0UL9RKOgHYIxsrNsPsuurTMoSn6L2Cm3YHXGBKheCjguUihIK4gOtNATHyeJ9nS4yN6fMSNeHVYVL8HSivMW0AkSo7mrREgF08JRE/uf2UV7txFdhUL+W5pdwi2uc1wY/NIHdnQ8wRjkEI1bYLhtLJB4Camfrhbu5Dqd41i775/WLM449p2fBdSu215rbAtYB1z2Q3LdlVAwaG/Z2jBtiujLEoHeaEiKSVlQSSVibKrNdZWm0VkrOQonqAf2Ci4JhTVlOkfI2emDCiFpqLI74Qgkt22of4h/FyWsPAzFqaLdcJvwFfR17JKTPSawCUhQChQBVHsGNXY4ID7Cy8Ph6++AYc0S4loc8xckHp42QdIKrEpB3LCZ0pZaK0rI2hNAoIPgyUBOKBpuaTUxAgnw21IEQggy/V5ITbtDPoPBuzH3x4UXPhNMB99NxTH1XtXyK/BrAtGjmOe4eXjbI5xiUBD9ivjs+LN3bn2doT0hIA7GHo/jAkY/v9w+CRfwsxH877LdY0aAf5r7hgrECzmHA6HmpvQ4sBJhd4PHPlt7u5HzAc/qC6PntRoRBdS28ltkuW41mKIGYQHWJ4Lx4QHWmjS8g8EWrrIgfa1kaYNsI22BDtjeDiOSiHKfxgqg2bSu+amSg3ZmAajYyB0+5ubQ+4hXc+f0BBcaMxMOM20QSCIMME/0yhZQQgzxd68Bb5FIxzDiSwN3F6mCX0/5nFvj9O0e1cBHvgZ8fnNj/5eZkWAzxI1TUUiGJCREBhtwBaxsklqoAFz2IIFlxJ+XxJrwvOKGdhkYeiQqWY5mcXdmwtTNFO5o27gacijwtcgeImFZfc2G8MiXoGZXFOQiZwOK3q0gQYyFoqj/TimISyyOFYHLSM4EVmxZjabCLzkohMvW5AmOIZJjxbSGPZzEbPnPB47k1ZLCebGTct6UsDtDnZjmvoeJBhN0euyqEShk0e3MwcPcG1gDsmpMpKOWdxYFZvEIpnL8UBjiisO0xXn7pgVw7WCvTzqXjRC7oKaKKRg5w1IM2EDg/kM0WMOEzVlTAsLVTWgQMe+Uo+awE11P3IESFI1mFfAJUadzmmAb4ECff0KWsAL7SjD3KHOJvgruyex3PcuwNcWP7W0dhi0xZzZgmMA+wV5CYkYknJAhH70uKKQLeCdgzySS/J0sEy+BsM+bw2YQ/lUoGDSDMMwNO1/yN+hseG5iVg9BocP/QV8MIAiAh8gfHrELyFPW9ELBChgR7Q4mCUkFEJTs/KFdtNh2K4vhAofm8Qe77lk1ZUSTPSngy2M5QREnFEwoExATKqOxbvJ4GA6OZjaBLY+wUhIYusdp6vmCpBrXRDkGk00flNptIkarjHNfsNEwLdlUIGVisFNe1b3lz8nI+o8xUTYxKo2mPrwTjVDdpqaYkV+6RMviqHMxKhyAciKj6lmB8UFlwDqsZX3kQpf9ndTbEIk6JdokPA5QrHaBxiagJMfQmQSJXGLRtjnlPUcUPhGFM5BlH+ZoAmbpZTxiKjqzcn4uHhNZGGRPxVyUNbrTCXz2pOM0LZFA5+KtiTkCc8FTkP6vM1oPFGz4v4495HQNZIoQakfRDkwV0DNzgU+On1huD42blZYbCJjtNOnoSPG4c5PWrJaBIJwh2/22ijUYsQxBfUcxPpYFLheFvVYK5u+KHbCxZVithXjQ1msJ4Qi35+Gz5uPivK5k8JMaNsHFbEjpRKjrC4SiBUlxT3kZGGbthqkaywyxdI42ezZBDdzQQqcJ4bgXlnL5QOt9ysUB7TQSWoMAQfow/UtZDmhze9haKd4hXaPy8UCsiwYQEmqLCmEun9hZHSz1FD8optpCuDqP9c8gLVWAuAEfBmPZzMh85KDzUQwByQJzkUM3+N4cvenThaMxChuJYGRlgAyMCjx4jqtxkRpIBpvz6m/b9Xbip2bGIhBVHiRfYVosqCRDCXLUihlPiOegh6/j0kjILpmjXr9jaYNVnZA251h0tOCxLCaI18omrZBG48KeAFel200xL2AE0J3nN6J5x+5iJAruTtme2IjAesGIh8YiDkapr1ulFCB/J/QHWgvixlx71N+41uLe6HmXBSS3AgYoQ2AUk32ezLqCGNf4C/jUi41FSzFba4PIT+ITa+t+gVLsCa8DtSWdZQzamy1Nz8Jyw28ipqnoWFFvoLIAlquF1BvAHwLgS0JtM+tP9DNBVb2X63RAar11jSKP9EX76ipJKsS4/O03mJGsICYAujuUK9nge/Biugeoq4UnDqz0FKBlgAtIWukxTWxCkzBa5xjDhiqL8jypgcjFLGlIyTDK9nPF+EC0b8bgNUZNZrobFJwd09QCTxfNHICNi1QkFUI2i/3zImGrntsrWYpy3mtVlTyfhANdRxCZvg5X4pyY1GIw2uOOzQQvjctzs22qBXLNj9sK1wSZSBQI2eBohhfvDPYyItsCsNz5oSgz8CE22qLLW+Z16vxv51xSxkYtvMf5oyfkEVZSDBWtI5H2n33LA4ErFM0saE+aYudgwdqPHWgJpmjghHnWpw8gIbXjU1HGi+U2pfCStvQ6BqHR2fhUA5Uo+u0HRyy0fAtt4HzWkyGsfIgLMfBOCvATwlMtfXNU1/zBlOkFqBWbom+6pxSq4uZ89avt83+vT4MRB2eYdy7G6RvllpsKwl0N1Xf3n+hs1khJJehct3ubbruhFiEUKUGJAReHOuSU1VfWaVOyhNp1+uJOLVZFiUi0kb39SzueC1C0na79kscWx0Bk5a31YJn5+Re3PhFlv9tLyU0p7Ke5aIosPP7MgFPAI9rYXUMcKA4/+MWEJK/omLnisIDKtDCJZkMsXSO6lphfUjqO41Do+gPwW23wF1eFg5/prwnXu+Hv6Q5pLIk8qCBYJdNZcmm8aZyDM5CUNB3i1BUMDAosQGIXDMOS5+JijFkBPhvALJJEtrBdatL52Lv6rSJA5x+U6sCS+3kByNSha4LOYuoCFc22eXHPv9oOEJ1RG9m9zpxpFr7vPqbVzGFBkuacqedvHel0jek/EU5N6i0qpzzKP6OlInwoxWF2n4gt5HjzclNJ9YpuIhzrFxPI3rLT8GOeXhAtw6NvAGPKXBvhrcwEwCWKybaikJ0R2bvdF6MDUsI6ygH+X3K+er2u6Sb3YFoW2waeJu+NHS2uZPYWwymFVqUwYGUyXA0NMBsXLq98dOiJQ7Bx3JU2xTmIMbyBLD1UXmPY4fyEwFr6rQ3Zj42H7oSozrwy2Nj4825HnU82YRpEFz4eQCiw6OU47Ta82CHCJLL5rb3LF5OdjtfUwgYdJrb21MxkqtisVMUO3ljuTTttnJGVmv5st1co6SORArnDfvRr22dJBGotXwwUqxaGElaciDCgBM1CeOOqgqzb990bPqmP6OFHkakd+aI1RQbOCKcW/EuYB72tS9FFIqvAood8UhxwAc3IjlPllJdUiJalaugtaZh2JVAafVqra7RT+4UjKRZnTavcdi5Twzp3IKgFtzoajbdZDjSKS93jnol9lxeQ5Fs6mp1/kNQG35PbfHCT3zY9b5bbOmdNvUW23qHjb3R1t5qc+uP4lBwTNHxrhasqDKwtMAGghVNanV6xmq0BHFy3T3hK9glGKRRRmFHK4iPtk13W0gLQSiozhdGfXw0l1IerbS6PnRMnWIqYrPTjvRHYme2GxtebUsrQDelQjqGsfaeLNwbWcc5Zdi9a+CN+2nEubjd58ULFW9sw3M1tNLmraNd7gxaBqKYh/u6i6u3HcVXkJyX2+Dog2mb0TYObW9rZJ/Qbp8ze9ahtO2O5ebmm7yMzT3MI/uwuaAUsXrKaI8d9o/UaFZNmBP4e+oRWbGdRF227g/x0wUcdViWXTdT8NWkkj6JS/go75ggF9usNHbhulzBw0HOlRFUqGUcGTbw7CsChNZgbpreDpgolLA0gYeQfby7wzzhBfIS1ee6mSN3i9tcoqK8C+SFLhF8CyGtWtJhRiwsMWMTFWwHRmcZN1zpYWyDzdGSkQ7TyVfTnRGUdif5paPv8yWD7OleLKI7t0J+7c7aFXCFalc9QL/3/yWOlNBQtDKaTraeS51+28lZbYPiwhgVbGa0DhLotTwChKQwrFgz1pbrg0IyZyLZv6udMGaw7cQukDMDRhKEOT8piGiomF9CYijPK4qaEwKyZbixN9W6dHc2bffujI9obJsAvtCRuL12+lS2DPXxaq7legnMCTes1GB6ApAul8vpJi9kMWmvpg7Bwi2z4fJES3nE+gAC98A1cxDIWICqLnSBC2h4rfyWC2iMGT120owmEJtKpkVB9Bfw8PIIzvGWibiTIuyyk42T0VDr8BlULavbdjNnryf4t2YBpOFA2yZ6GKVwzB+QB+9OEQlqbobR8ACK7KLha/z7GqqUZJFB2DqtbZLGtZs6GErGKGJe+8A51zAsNrA67yziCaKMw+y2KfziQoAykZiXibTMne41VmxRKwtWOUktUYBfIPhF++UtHshIruAoHNVlMKzXhIvcGCAGES0ueP4iKkKsq4VAQGWg5ljUkU5wmY3WxGY4HjEXF6jxulTpEHVKI7WioZtfbTtj3za87c99CxL8nH6bm55q6u40c59o4j7ZvH2Gafs8s/ZbTVqlcCF02fBLnbomaW1mXvbdJjoWLo5MI5ZqGJ3xtqtOvme6DBIDbKs9aGkIaDasdoZHuwEkE9pPliJ/tDJQpnnMr47hfqVqblwkIxt33yaD/i9UWh227pXZeEjaOiAtK/RN69rCTj/KCge8WzGGSxZhh2CqX163GJ2p81ji6hh4WYENrhocVVdrDEuc05eAi/sVMsw4jtNyDtcHGj2jJE1xGOoS+IMB2jEDMi+xNgZPiovKAn7eZ9y+JG8rOKp8IwttKzR1U95u3PSlDzvRkzfobQUq5MDAPE69FbRxvd5WwLCbAGi1HZi+Xm/7zIlzBsjsEh6dgZDgDodbu3Pb2FoHpzvewXh4sBmIFhKDAdh0A6UuBy1bUIE3CuTsSXeXtLS2i8DD3Apid2BZRuBcXIUN6N4Ho7V2vTfuNzomvamayC4lGiPUSD/rCdo+FW+n3vVtUHJgA5qSPZpiTi6SN+7wy3QX7fzzxiV/MEbXno8Yu8sVMtwo3qrTr+oy63nzzQa/zBNkRbcE3hmYJHT9EY/I83xAZ8xIyAfe0D6N3RWK5WpLV/vbvuD0acFHG0Rng2lXSFL3ERaP0+rRyQa2dZMdCpZ6yoln4cbBC2Zj1NtwsS/WFMQxbqM4FjVafE/1/g/yZUG/clkAAA==", "sha256": "d04f501f1a25cbfd13ad7e14fe088652150b07e12d8a2bee49e6225bf3a4a595"}, "run_cfpb_v052_blind_semantic_judge_v01.py": {"payload": "H4sIAAAAAAAC/9U9a3PbyJHf+SsQ5MMBGwp6eL3Z4oWpKJac0u7aVmR7rxJFhYJIUMIaBLh4yNbq+N+vu+f9AEjt5urq/MEigZmenp6efs8wDMOrvgq6+zy4LYtqGbx6ffnX4OHoZXIStPk6q7piEfzUL+9yaNPU/d198G6TV1d13+VNMpl8gI5NX1V5ExRVl0PzusrK8jG4z9qgqoO6yRZlDn+CRVYti2XW5QcIbg1Ng6y56/FDElx0QZNny3ZSV9C3WK/7LruFfk32ObjLAXqGgNspIlo0wabJN1mTL4M7QAnAVneHZXabl9AARgmyYNXUv+TVpOlvm2KRBO8YFu2ibqBtULTQpARUmilMEiDBx+AhKxE9wLTt8k0yCcNwMgE46yBNV33XN3maAmabugHEq6ruGEqTiXjW3AGkNhffgQD3ZXErvrI/8CBZ510GA2XizU9tXYnPdSs+bQC/Vd2sxff2Ub7qirUcpe+LpfzclAi/yX/u87ZjuCPBsb3AXHyfEpRf6ipn7TZZh9iKZpfwlb3oHjdEMvb8tHqcBq9ggXF1psEPBdAwKzmdNo9Lxi+88V+zNn9TL/MSutTVqrg7KxbdNHhd5OVyGqzwTyqpPg3W2FQ9mHTN42wSwD8C3i6aYtO1ieKGpMrXdYqUTJd5W9zB82Sx2tymwL0n6abY5MDQedqu6085PDtJF/V6XVcCu4hg47/zh2KZV4v8/SarpvLp+/M3p28/XLxKr85P3797m756d3b+XnvNN8d3nJnVm7LOlmmTA68tW+up2FGp2AJag/Y+O3n5TboqgLDy4ecGKJwih5TsYTzJvyzyTRcAYfsyf1t3r3EHnDdN3cyC4PfBWQFDd0H+JV/0SCXcevn6Nl8uYbu8qmHhcL/i4ieKuM8hGw4CTJHPAiB53eTXVX0AWzFf3fw/Jehkkl5evfvu/NWH9Orduw/p+dsfgznsxCSvHooG+Owu76Lw9cXbs4v3lx8/nJ//ePqD0SOMJ/pX6Mx4CzdR5MCOYYO2dfmQRzG1KlaB04ZegDjLOQyaRJpqXROUfzDd6xc3k3jy/s2778/F4AYuh0EIonrTd+2h2jmHbHFR1h7S0rd5vrTWv37ImwaWMZxcfXwrQGvjAGDgo/Tk6OSboz+enHw4fvHyxdE3/4TmSJ+Ls9MPF+/emvSweud8q6MUPZTbHlnusK37ZpGn/gHiydn569OPP8AMT/8LgEv8EGj2+dCQCPStzbtDoBeIxe4AKdke3mbd4j49wn8JfxNKsJdX55enV+dnFmyhdNKiInqS+PFQjhH35CghLtPBvntzqXGHtUwLEpHGMpHebQ/V7jTYHXfncbJe6hR59/EDsKhJdmdB6CmMyMHXoM8b0ufpz5/z6kX6Mn3x8jbNXuCgx2nWdfl60x29OFzAGt0y1GB6+rBvYCv/AAOGCOCQoCQvDwDKAUBRFPjr6fvz9OMVtbzvuk07O9RGT7LiMNsUhw/Hqsd3H8/+dp5enMnJhCPYLvMcdNWqydJflk1a1VV3X1SfQHulZNqYs9HRh6X48eLs/ArxkkAUEu9P31z+cPH2b9ju9cUP52KiMMe0Xq2KRZGVwBRt1/SLLmXrB+v+7vL87RUsyPlV+s+zK9jWZ5fvLt5+eL8PBQ7zarmpwaJqD2EuuKn+/vECWFJCSYFDT9+cA/D3AIuZO8DmjEhPUtzBLsuAC4EGoZKB4Tr7knbApFWrP0XCIOeBpWM8rjfpxn7wyQBXVGYL2Cktyv8UZpaV3aP+DjYRGCDEQ563uJ3M1u0GxEOeoimUdUZTInePG5JLOP52O0GZviiztg3ed2D/dWSERNIciZlZwewNtu2AhspEifIvXZPNQxjztliGCtplD4y0OOdL4wAEi/FHJsZA1bb9LaxHUK80e/k/2mBDIAKxvIEwBhMyN/dCi6ldRAubV9kaVDEQQ+tcLNWTTVOjNob9YDTssjv1pQVbtm9naMHT9597lDK/0E6nVsF/B2/BVAR08A/r02/QHgDqo/0M08gbgFAWbXcNHW742LA3KhhoCejj4ykakDcAhmzACIyGrC+7dJUtQPI/zrEZU4r9Bi2UFKjepS+O1rNgBeq+s9AQyyIW5LLJV2Vxd9+9r7JNe193kbb8fI02ok0K+q2l+XEj9joE+RAicvSBTZLpITCsFbGaHIDmDzDxvlvsRfm27O/0/j/3RTNKtzYvwYaDJoJNZhbrcfuguCvADFcz+ND0OU4A/0rqoD2VC4vMQxE2R40M0qcKp0HItRHuyiAkHws/gPeThwzXDriS8YhYVRQHZV7ddffzY7DqQdbwb98cHcUmWld8e3vQWuaLwlqfbIGWL47f5D8BgdinhyL/zHFhwk4n5yCrYQvGajknDe9lkGtXfwLwF8uTibjQbUM2wF9owrDS9/WST20lvE2wcVjbaIHOqzuBODj4s/o2k+Kvrz5V9ecK8GtpF0Yo/Hl36OM3tmPZHYxODkGBJApmBRidIMX6nFyKaBV+5CPJaAAbJVjAYgGmTxzONlTQYYv0TeXDTJDM8vUi/D4PsxWsNIdjEEkwA4pFkIEgC2GLrIg2ocFI4UyfIjZKRN9gDjubsxBFCeitoPg4FXg/sXchfFAFaEY8ihWjJQrj8cE519LgsIOehQDrqyHQBSX0BAUD8pDB+FXDS+7/LeMLIAHs1LWHD2Agc9+/yrx7noxpLkK94oSBZqEerlWkdABxu+yZUADdm2pfi7btc/GcfbmZBlLaSnmnZLSUerbqktO4opgXU86emTD+lioXJmOYytTmFoggVYvWQljJ1IgZ+oomtmHM6JatNyXauTBp9G80mbnbTr2RtjX+G7J5ua/MJL60E4VetqTk/Cj54xSICH+PpsAn85PkiPdFY3K417dGr2O91ycyT5w+J0esC2tIlugw+CM/eNtcHQJwnLwkAAcn9rRco3YYBiLRuZRRNvnQRL9m1IGYRjxgX2j8IVwaagnxuvozqK6yvM0Wn1rFHq8ziDGQksMPunliWCde+4KENLrai7pEc8XU1SAPHpkxRZ+oNXgyI7CkpwJWD0YXlzvQlK2/LMp+mY9ARrEiiCrCCZLm3CEcoPoxkfyYlurrWBIIwqspGqhg1YNPDkpp2Q4u+MmRWvJfqf7s/a1pP10Oacov/7IhCxLQeTIku+HlzQLcqNZ7cvfwzbeeN5/gDWw58wXz/7CL/cZxBGcBbiKzkccjxGYarK38BKq/z0qcFZOtEJqDBWyIIFPyheIA3Db6BAsq6bDVVSMH8ru5fL1D+xlvCeedYlVpytDtra0BE5VEdCb/iMok1oisTFB4YNikZdIJdbtLT5Q6Jog9dPQp2xiMAlcsZExflLd7Ba7QI+VphCw6EA47hp3bxT3YjAfCZgtWWVHClJmrO0EOBw8K4jSfI+JlIOBsoqEkUhUJthDZigS6xEnR1iwgEMUcEmyjNcTFVHg3wozGjOKnU8huPGK42NbmNCy6ktw3hLY8rpqsPy2LJuJB1jmKkilwCxh5af2JvipdCCmu5hFYkrp/Lrr7tO2BLb4QBgn7HPwhCJNuvQmtbglDGJ0oxWWIfrLs15s24ojD2FWLGaisXRTFnMTgFPgbZCtohilZ2umn/JFhGqudAxxSo2kzD/tudfBtOLX0OEMCOKbMwDlEhAVBUyNwQDomesAtQZrGu2BhmPwE3mm0uM9wvSEliFtRfYP9SBAS0EZ5A4sOe1G+hTXNyqpfR7HAYJVjjBZDecINTqX3zoj1FZsOi5XMDJuMvWGBoVRjBTbzIdH9gkQYTW4wqCC5/zWwc7Ao6xbEbF9BVJmyisVDHkC4T0V4eKCkpRxr/iXjVjSoEBXz4Q8AAzOHl1yxv4o3xoKKatnvIZcKmnv+FH5sYVOeglDqQKiGr4vqrGiBJODhZuWhcOkoI5sfSOoeYABkq/MKsrWNG3zF0GXEv08FWecWeeMAUsH3IJlKzcvgjA0zJm7HzxFrI7Mi6KOA9wKCNauIOflWoCgRyhd/A5auQcskjKfkl8VqWNexUdE5c9mkIMNENrh7RUXqmAs1EX2TYVqYyLUZo0kMJZ9HTf2Z6Sb4gFuBo3vNUL1hvohMkhNAbRtrcR/K2mFWT6CKek7goeu53MQEkAc3kG2WRDkoSreC7vDtegnGeBdrwAxLlODdGBGq581F0cA7GRasxNGPDNTHQuQJen0Yj1WT8cUvYw13zoAyyCbHWhYZBGBb2DbmGpmBd6mnkVaw9eS4+H3qNmOz0huyJ56ma5gMGYMSb+jGYy1Oa5YAGkkeHAR7ksUwH0y0tsa3PdbzxtqQunlhziF8WzOZOrRPW4hStxCBY/IVUjcViOM1qzoQfGkbUKuQ2UlM6OqezpO2ulvVie92EZdVUTfBHtMAtO+8zNa3yyxQUVtJVgi3x9dHPLrLFQlAGY5cqyy2DEHP95P9Rnx6Li0s1UBIg7kjClQbY0fPfdtcH9AJaM85ecY4T8PIiXfPxRNdB7kmnqbep5KsfD5oP3G/6icKjsW6qSIac1NDelpj1oZtRey0Psi9hOjNPSyhZmScfL2vkcFlENmRRUtFAJGjzF4XVhkIN+P2ZTVLTzHKanoaxsbSLGaj2vZkrK0PBh1NmKBxfufROGPqWKCpEjWMX0HsrlbAWbxepW/QMOegQ3d8g1U1JMyYyfPwEH33QEVKp7lCyWHyyYBmA2yPnokayFcR0sUoLTdDBeDQMKn+HXrymeiVGHWScsIjeMPYTK0B3aT3h1RW/p4kpyHmuIAQew0jmsPeI6g82TlOOijkK4WpCj7JYfAC8lSCYArkn4IjtDnVgz9b+3uyX/hg5SMQFCTC4oOhFD0pgMnJahvQx/g/AeMVWKT3EGHXtdKwRKNSxBS0WVnf9bnw3MDpJbljusFS2GjGNHXgtvbMDhbQy+FO6B/qln7T5oah37KGRrrCsOOxgzO2Nj5rMfHz3xWUjIp5B+se+Ow2hx3xHWTBgvqWUiDCw6QSMlYbsoBgfBtB+VBqxA1EyY96qBJyMpNxI13CVyApoZSU1csSSPKcQE/RWFhI6SmIZaWQrXIHZXnr3Kh9iwx0+OrDbK1WYhKxASu9fSQPIHiC5UFn5Drk6ZbwJp6RU6I7J7zXVgzyvO7QYSv4A3I3kYlEjCJOf0zeHD6QA8km8GRU2pz1EBldIBE5Ojg8oz8r09KUA4g1DyKUnpTjjo0FnHUolwaHA6EPVQSwYyFW166xpEz4hbT2M5tP0F9gNiDSSkOYm0sWdjPNQCJmMN9fcwDKmKZRk2wDprLlCqism2PLcyhz/td1OFi6bf7k9S1kiivC/uR7kyUJuyGOvT308Gz0vKydC3DroivTeXMDq1DV1aWyCSYP8obVFAC+HmAiETi3BKrkGLkKPO7AGzjQYr/YpiXj4gjsCDRN8FGE/83UspnxNhU80Sq9sAduxxnBTJzlZNUbrXjP6+XVa5covKV8oTcWdJEUka3FG1GMpU1WMV+I8acety3U6IHyA+glk5gQSkMbC505KHmHGGUgivivGBAmyKF0NflX9a9KacM//Pa4qSwINnwCkd2OREhoZhavTAPfYqmwMvPgWnO1VJHNPlSGRtrkLDp75zgYEeZFPC5AcYBib2hbKckw7U9in1PEU1WAKh5aJYwYVI9ARUSmfsdUQVH1udMNnQ8W76sEPa81eDfPTeCAFSZrFloCjBUdFJx90uBuqYyva6g44Eli8rtma6dTGM+sM6j8XZQFsFf0FWRjNgVSjeTj1OsoEquADcbwlyceZGE+BlizQtTeYwzk9IJ5GOwUwAW9oElilBWe0iGADSildTbDAzikosDk5cXsWCN/0ELWq4DE1VgUJrxAe6wsBQq3Oaw0O+uDZgzuT1KzYcwwhaH1rc5QVYTnpJjzv4ofRRWECDKI76qFCCvzBgP5V9UezkDUtKRryL8WGzhPxCLvaCgUnchK8bwvmE71JriF80oFnJCogvdn33PvoE00iKfK++NZLLLZ+wbqwUA1o5FXsFNUDEssSxWIKzjoMHDg86OpIW1U4BWqrEszvUH8RMb7rogDk5gIC6bG2E6aI5qAEmCaRuTtVY0nj0xInuxbxKmms10gwmwdJa0rDZoqUtP7/kEvMyFJhPL7H7DP0K55KOq+VUF3+FpSGS2SFCX+IVu2QCQTEyek94ohAPJC2tyQXKruBLcyGjwpPLe2J0X1teltvXz0VKw+6eXNbKlAiFrJdbC/6fG1L9xwY+W9raoMVASsl/XCSZfbBRqqp/vO6mwVbKie1gurG9agy6bwRUvNT3215g5heEmHjqlV7WGNyKs6vB3YKy8Ovw/+Dpl5qLMWiflAJOYP4TzAgTgQEPBCiiSA84utHlzWICkiwtk+9C+qTkoRECptTWwFAhEz7iiewaELbvEMJTxfJxOncILPhL465fPyNX0dL5jXiGK/01Xzp89wJrEd52UKrmmDe2LB4RpSm+DkIJ9Z6Y6wqUtco5AJHjSj+XbHlIQujCxDXfVEGWH206WG1u1m4KiCoqt66BxfMIi/GTuqMFOhQvON//yEopx8Zh9nkE3wm3mAIc/W8JoZWcOHHpzdhMfvkHZowqU8IWntH/3VbCQvFfLzX9YxutCbosIqR+hD1RCe92I0wzrmLKXhE9nZI780UdIYs2Lyi8etYAoyAWkP/wGzlTmdKUsWFI2JvvqKbQSpatHoTesNO6psh8ak4itWLM6Fil4Z89qw8igEtIQkMdUh8QhYqHIQ4UD4bChNMRxWi5Z0fBZjClNmUUIQccrSCtPgtq7LeCxUJ+KEAktJjgqsqZzq141A4VT59g5hnhUolEEBw/EVlVu8PwsaIEklXisQ1OCHNpiKYdbZPmFMKkvCQJgs6zxsi1/yQzyOLQ0DwCEryI7lZp/ULgEXPCoYt5MPnrgQ6eRWhl0FZS0oHLB2i51DhS/Ye8sTiQ3oJkBraRTfqsUaja2qVDqeENX8OB9nx3t5dBMnMPFky0icHjJZpJDX3UoxYwza+VtIOvCT8Ql7oLVOKL2URyK5FCf3+ZdlcYcVMNLp5NwhhWTWLwueoVMOurSVuQ2tV5bvMqK5Z5BCMdJt3lBdKDeKffy2uK8L5t0Ljm5kXCDkL0PO2Rjm5AFA9gZ68SaQHGblUAwarawUMFwBa2OwdlOpm8UI3GQ3G2sI9U5jSUcSsf5eTFRRA30qT1ttKk5/heJob5fdRFx5FrgxLHNtoIn5wKdECZJnUsVSYKNzMamMgR7ipdtNWFFDtPN00ZwITy/5Vl8qqmhrLIFtrp/e1dh7zBbTRKp/XE87F3V4W7T3/OSSBkeuuNnABVBBGPYhT3fC8bbzUMQzkX27msTUWXkIyHPWwg9A5zdp9Yo58A2tWcQu/dguntmWDD2OBxwynzp1x9SOC7NRveDgPBboT4ol+8BOzAj64BgSjL0u8dgxX8cADXlECPNialON16ygR5uKhLTHtZVqhkd7dC0zqFK8imNc1+Ts3o5ziuZJv9vQY2ZcZiBzKwMnjpFyS2GPERdwl8x9ntz1yF6GvdFAiVnPSyFMPa+09fW89Qo5Tztb7niaDEgoT0uxFz2vdm0/a1s/a4/5+u7eKr92u+yzZcxCQBUxlFYM2jQmX8Jbrx2nDGwuzyd7JywZVjxWbL4yWXY+xMGxZw6IbLKpN5EtlCdaO32ZcGaeVeMtBwztr766zYxAAJEE8k8InG1OEKR441OC/30dGdKZz4cVxGAYxa38k9tYVpuGKN/MKxdIJKU8xIB/InoSJymrCU6dtsIGZfix1tezb4+ObpymbOAUrXxN59G7qSiCZW9drccgDHQd7SNSBoYpKHpqLz32IF9a5QJ5Fvtaekg3nq7CMfJ2ZC993aS35O3H3974DF7RgTm5tFfdCJOQMns1ZgTiFW5WrNAJYJgKlJ1fBar+u1Ma7AEn065UB8SeGKOL434zeYnZ9bWpG28YE9yYd01w7dtBtUl+7dxUZalX4Rl68yu6RKTrLWwTwNMOc6ymuGJFNsAQ0fFUj0CK04yQDT/Wa7CQa+w6NFXlK4AHT+Yo28MnD+ytk3qJBKD5k0+PbKd49FO+g89btyKbRRf1OCE/dSXSf0/j+b9t68JUMdn5kxOm3cZWkHNV9u393FSY+jk5bmFp6+LkbK2GvqSeoaxoG8zZHzeiOqLLnJ0xN755YMFemC+yNvcNI/h0rj4OFa9ohaQy28+DFjMHsltCdSk4zjheg2lqHimJnXoAkXc0B4N4ScL1TsJb+XDUAmi8FQuhieNDItqH3LeJ4l83BYEl2N50KYQ1CVkA6YuLm2XXhn2gRQWtQhRRIakSsFpJgBIkvCBgYhZzCZETzGXO3uUELFxGHfllwTW5uVN4Cnzu9Y98tpq/JOmZltme+4JmNwfU3VdiyeZ+y9Jhc0dviGoR08eSAQG7ecTJ4wC2pPmffDJ85pu4WR6jONQ9sqsWU6vQUGdMxR2sc+fixWjyvBXkN5iwCrmluNDEbCNYd36t3w0JqRmqqdEsiTg2Koo4TNHdSqGL603E0OK72YqZIJD2Xc7DslyHvrfqNIz4brbi1gbYkndgefN7/sj0mBuGiNlLZVPHztowGuI1RsNndpzz2j9J2+M3eFDDa/rvcqOGmHPFru0J2HaReZkMEYaqHchYKkG1DQcPeZfGKW8oPhY1t5ZB9n920FtHVYJgmO51CJsKu5kbiH0iqw5dTdNrgernh2hObeTm6EQcHsA79eaqzJyWZe/yb4Bl1H+z7qqoG6f/7KpuKLlV1wsQTULDkWaDiFpauDqUtrLhd1j1/WaBvPuCNrb9WN4067xB6AJB962pGdz37rEzdwCqM3Eej6Sy9AJE82o5TljjsjSfy+MLOLqoQp+9ztCNTHQ6GRE+sesLUj28XKGdZ9QMMY3mjbodONLgxKp8X+wG3wER62iIdsbAJq+6eRiULDs6mtP1ZWY7JeDFvW5PunRmNylgyKzgNePb4avg+EgHDNRvvPSNn3ilEwgXZwMXvgl6XROKDq6U2dRnAy8Ykjf8GD0KJ8y+69nwgQueI3MD0i0S5iMp61galQs4UePHbnxBmu9L5L7SrtPRLrUTWOOJMh2u5AbVcUTYQeViBv7uUk2Cys1AHbZgabD6eJBmRHh5nQ2nPSIsLTm67oZhlFA+ypD2gE7kGD50+b6yj1A+k4lkNNTGkNaRdqRSPBrso2we72nQwX7DNhfCMfazsjp23WNn0ZpCIFDfhYXg3mOdvMaZsSlTxOIklKOcXfkfW/zDWptcpAHdg5EYhFF2unL0JHoto/zEoAquEhcMLIhWPpx1zrfnIJH2gdn/lKSzKfAQqccG4Fbfk280p8LX1MEpt3ki703zkU9hk7TxvbBkDq/ycNaf+4OtIqcc22dZ6UaVB/etxS/mYWzPsPuT/zWNJqvk92QjMY57dQKdScUbd9vO9pFhJjsuZhpYOnHSDW9WUQcNXB/Bt15TH0BxeI/icbBE+nEO+wCDaaXE4h7lhmkJOnkMv5pQdxBmX/Cb/qv8MwUCcfWP1OEZKKj/MpV6JwdHig72UHl9O2VA58emKLe1qdiQM8trxbDvKrx+okEgnIvWN8GNtzcBbRWogOtMNbj1BOGMqMMOsLhlUZcPAlW+qxAtVpZgj/joiIO6d1zUUCFjfvxA8NQJ98z9HO5z48WCXetEuuGkWOvRTE22Oo3p3RjrW9rQ1k/mfK71oYDHb4g/4YN2RFWX9De+ie1CwjTVLAQkVXD0oQSTByupj7woqX0HZ1COJ8b10kZ5PLA11UaIa4JZ/Sa/N1d8ofum4cv2uRaYHPJamjji0Q3DTPl4ZpE+9/v4jeWYtsULuvRisr7Z1C3VczNvBYw4/hsRS0gfwz1da0pFqh9wCN2it5G8MQNKB6rlFcxuWTpqT6w9oBPubiJRbyPPviu7fqADOxuf8uzrtVt2Ls69eirS+YlXzxvPCWBPI/eQq9nqxl+5/qxEKT8eIw/nuWTdPEJwrGIHKhLOA3ZdDP+xKGgkPibig11uH27AkOLHOdxzAazm2/2xKjGwVsg9fKibHRnEUx3iJ6HoM6x51rKnGdgcny09M3AOgB/jd8giXHFediBvPZi6zWQuXXf6hzoYjj0Hbjr7Qx28w+zoKgMPciQVivA3lqOMKKxwIMYiSjT8b+PdQLxT3BOccwGOdtuZ6urek6PvH4fxtA01GbihbGscyLHiLurqNMPxMs6h8OdoIfICd2braI2EFNMbSaVkjM/DGeLmiJQfWKHqGcOzms8N214vnwV9ZmSWcUSp5HTUS7gThiZLKfIQA7Ug9iLbOkUvjpmu0+CFjq52737PRLRUYlor8fMrzhbVZbvnCJLGkVZMx3e6yMd5o922HkkgzZ+dGHmc+X2x2tnVxsxyUHbi5vVMPdhpzOjzdsbm479JUJ+lFwlvr10us7eTeQJi7PJB/07f1JBBeHR5csAeoh9D89dL3vewZ9O7ulwOmCjgDz1ki0e0z+BUfT7U7BYC0/frrPmUilsDh0wePPZXwg+KlXWX0mHgQZjar2FZ5ZmeiUiZIY8W74AuO1ing4e7bc3jeTJd8cxsm95zxw3JrOn/ygXJMq+EIxj3auFxQnZptfjVz+QtWD8tmFYyxQgPscZCNjjlP3Z6SW/gfnj2i5aYvk7TZb2An/fTeibZcpmK30eNwoMDECpgOmE95pxlO8UF89qP4Y0CEIbIOBTx23c7QKHlsQsQ/trdKBimNw6aut4BS/tVu1GA4qyN3Z9+c2O0JxbeHsCtE57O4uc4RvuzC5SphNTuL36pYxzz7MuBFP+cFnS6U/6MwGh3tGbY6Bkr5ApZaIzQabt5KM0dg7M5NJ2n5UUmcI+3FYXDBpT91lprSUbPb2ZqF5aeXl6k35//w7yKUKQnR+8hcYFgHFJKuli/6BtZie25NtEeaVlS/E1H/adbJp5aDerOrpB1rymhl+4lJbKMhN67RSR6YQ8fQHuiS50BaxrJq03ykNXje+xnZQvvvkB9JJKm3a06H83YxmbYwkq861l3Nm/4pt8zq3lH7L14ZFxGK3wi0cQM5plm4Nyik5tC5L/MOZ0MR8ZsIEYLp7/PvLEhWG0cGANEnu9Ml2vK0h6T8SEuCWvEmGO6a+0HL+tx3Ce2HuKbzhAsSPxbFTXKI/yBXH70gO6NSlOUTmnK745iomryP/eSjcTJfAAA", "sha256": "47b7787efbe5bd94bc0997ed103127d4e9f1a3a5f0f6e68db6bb7363238c4a43"}, "validate_cfpb_v052_pipeline_smoke_v02.py": {"payload": "H4sIAAAAAAAC/+1cW3fbRpJ+56/AYM5MAJukLk6yGe5oHUWWs8raslaSM3tC8eBAYJNCDAIcXCQrov77ftU3dAMgJCfztGf1IAmN6urq6urqujVc1/2wWCRxypzbMInnYZnlzu3uvrPA36O3Zz84F4zN0fLNeN9Zx2tGoKNilX1izjwOk2xZsWI8GJyHd846z27jOcudrCrXVenEhROvVlUZXids7DiXN2ioB7nL45IVTpk5YeqwzyXL0zAZ5ExCxFmKAXIWAfZ+6FQFYM/u52FaxhEnbslSlocliCuiG7YKiyEQzZ0yZ2FZDAq0cNBfq/lyxdKycMICw6yTOIpLhxGhacQcYLgBxeUNEZFG2TxOl3hiKydOnes8C+cgaVklYU6dc1YUIKzAbI5vWX6vZppjtDgtwIH4NozuR1WKt/EiBnGgaXANzDerMP80AvOSeBkTQwau6w4GizxbOUGwqMoqZ0EAhq2zvESvNCs5E4rBQLXly3WYF0w934TFTRJfq8dfiyxV/2eF+i9nYogoSxLwkhCqMY6yKgXXxXtwnJXxiqmX6nno0O/fslTiWYNfGFSBneFRvCjv18Q62X6YYs3eYYFzrKnsqRZPgvwQFux9NmfJEJSki3j5Jo7KofM2Zsl86PyspeA4z7N86CyoPdDiMxiU+f1k4OCHYy+iPF6XxVhKBfqNU7bKAgCHwZwV8RLt42ixvg4gy/uBEuWAizLa9oMoW60gdJI8j+Omnw95GCXsPLsb6qYLKV0/SeGq39AqBEW4YHVTAiEKMo4kiIrbxgtIeJbPi0arEt9Ai28NUNyE+998GyzixBiE76aARk9Eo/+lvIFwQ7xAIvGowKbfxqg2h94cvz38+O4yeH9yGvx4/uHj6ZuT0x+Dyw//dXx6UZO4zCFwtL+CBbYoxL1QlLLPEVuXDqShSthpVr4lQL7uE8f5s/OGqwHsPxZVXC8IgSN1Iqb2VVGrinE97y9ZbRoHIswmDriR5WyaZqOczdli9n9HDr58hZ/BlT+69oNBcHb+4afjo8vg/MOHy+D49GfnAApszNLbOIeoLlnpuW9PgPfi7OPl8fHPh++sHq4/MB/RWQgmqSavhdsfY+wsuWWeYEy8cFow/AVLCiZxcBYHgdF1THzEYkxfzQb+4OI9ZqsGt2jZcVxxQhQ79ebbEXzGCt3vbFuKDOdHjiPKHZx/PFWojXGAOK/SYH93/9vdf9vfv9x79c2r3W9/ATjx5+TN4eXJh1ObH43exjlb7Gi1Squ/U2RVHrGgewB/oFb8/PAfQK7pI6Th3Y6lVPhTwcod8OufFStHxMli5zoso5tgl37G8o2r0Z6cYpkbiKXkzoM45czk6murBO/vjvkGaOAM3h+enrw9vngKeQMbbw2wDeMFK0qOuUZMQvnh4oS4XQueIQG8AWMoPkQVN1l2jPPdloEgrOYxuCH71Wuw9+py7+tvvt5/9cuOtDKCf4Y7iypJBA+wcLqXnoGCnMfFOitiGg+Ae3IS9VJKmTl+Q6xpSRAwSgEB5bXR1+Dx+THNejuKnP0KDd2P4eeT439s738bs7u+3mcfzi+fmAAxAMsN5UYMk3wY/Bm2KcN2h3FDZihMxRyGEIyjlDqESXLvpCGOoztnzgCwitO44JYMQJZQuPcOLNDoE5mFp8fnQEeGqLZAIywy4GED3jvXLMnIwMycMyiTeB5nO8U6PIIhmTuhE4HIa2HTkr6HjXh2chKcHV5eHp+fXmBqD3yFXbI2E3cC426MY2yNPeXl7tX19HD0y+7ob+PgLy9Hs5ffq0f8fzWmh9nD/vDx6todUscTX2hgd30D485GptV27nqv//6nq7nvvZ5cvXy9Nx2Nr4rZa/81PXuvr+YPrx6v/NeqmT/LB/z/9aP3mjoLuVTjFUXaJF0OwruP8Huf/9bdVc8q75j0TVmui9eTnZ2ri5ebq+u7u7urMf5tTJK4jl2wjEtSa9vH/9vQGPRxcIFDDJv7Z+jN4//+KFRHk09EBNghnZ9NwdL5BkdzzjYlSxJnxTaMzOxNREZuvvKvrscPu8NX3zy6VvewKO5gAGxgRW+i21vgga7gogUbeYMlmo6cGbfQeQP2AABp+zthFJElv1H4eGMUAleRRdgtjkK1Aet9Wn8BCPYMoQPenpxfXAZnx+cX2C6HR1KVbZnlyeaOkTB8lSQb5y6m31GYbuCL3LKNE66cZcYdqMy/Kl4oggBPen9TVNeruNxka5ZuYnhI2ENLCPsGdliMs2ezyBn7jW2ukyz6tAHaiCUb7HoGnwd/FzAg9Bwj6GugYgW2DKGQJzMxuQyjsj3L4/85e3dydAI1cfLjf15eBEfvDk/e2/OkKd5nlQNy+XRyVsKp88kbdPJ4eVNiWvXuGbw5OSctD23//uTieDvLJNc2QHNvsc7fYLSchpOTE5PyYdhlVTKXgvJ1Q1AsWIM9av6rdQIdtYG6io3nuWQgUz1ZzUuJY66QzNvcO/zh4sM7GF/ds5WULaswh8ZjbIO9sIpBFQxFqEpsg/tNxHLiZnLvGywcXFyefzy6/HgOk+78+BACSEoOtupvLMVpKdj4oLWRi9UlUoUaF14/jmeu2t3hFrCEpcvyJgA9K7I6TLgVeZ+BjiMESQj1HHBH0wSr0k9pdpfCdCcsBRmx/LTWgz4OyIyNEmxh5z0iBOGSedq79YWTKsbiOmCJOdYer4fQRx4euAhpXAOjsErzLIHNLf3nqYvgRw6muRgABwl47Ar7m6QdymXiFGUOpNx19nA6yUkf7NV0vZHH5h8kzGTtxElAzVTOWFBU3KeQc3JYIowTFNUKYY/7PgKplzJTYBOxQqJFj9mAv/2+4fp71gJLwr7n01yx8iab8wbIngo2scDs4EUJQkUrQXXRmITvjP7DbploQYCrALI91dN3QC5FiR6+HjrfDp3vHmtQvoYhtgDFMSrGXVmbbGdVFSVfQOwKR2CADfCdJkzOiztveBFj737WZNOwLK1WXHBrimwCEK/iBhcYLySIZsDxOH9x9p2DA2dXeDmGXFkIAC5Rj0kgnT8daJz2SM+YLiEoxKTDhMf6Sia8Uk5aPVlo3SpPNRsgwLSSPPAVQCaqiLzHuXdLA00ozsSXbA6BJZEZUotcMw6CyWtXXHTSfl+MrUyzjuSLIcdiMFGSwl9u74RRjT46KqV+OOVzScaYnPrCpEMsFA9/cICfoATfMDriRfzDFilB0MNjk0Q5iE2faJSzEiste9Z4BHfJ8AjSZR6uCq/Etuf7dehgh8PCBenfcR5DI0/LCseJYPR4PJ5JTlP/QpwJUPhzGMw4E6bh6DeYn1/NyBYjrJj8Hcs937doqDlHqD2OaiqkdCKl9aWTzjp2A20CHDhQtqvws7c75LuT9we5Topue3IsNU+tD2DKyzM6u5s0hEfodK7hr++h5M33NqQGbcRoJq24kLNxTmHCgUX0Bycq8VPwUiu7JnYuWZLBCBWfC34hug0xgQLTkfURIu446ueYS1XyP3ciEs6fxzzKLBjOexrqFeRMZ/VsY5JStHvAIKIu+pgDAZKV1ERgmj0cUgLqrcWhEJ6j2Rr7SRB4wCdKo5i7nhNHFPWdt7Om4Gt2CNzD2s2UsyZpQey7WqUkLh6cfQQZo5KmBHM0MB4xVMVUu3jwLeVfoyFPkLt4BrPEW8Ep0pT0jshuvWxuaT6BcbiGVTz3FoLn5EywlHZxEN2QiM8nDwLHI1SlWAitC8G0lnrUS6jcZbV+wHwTX8fa3Cm0Vyn8JG4CGVwRDQ1WiUbNsBpGNtX4vojdypqS/G6T+lfSQV49TUtXW3zste0EJyxNrdgEbihLaSxQKJ1hDjswtHYjUbGdoj7j1e8ihDbPoE2qOoID7vUBDjq6Xu8O21ctvSTYu0RAVyZV6uPa7xlgtKeWRFMn9zaZLKR1VfvYHN43DQaB7JncadrsUuTr1aTjhMwaYa9o86XR3iCYzDVbIbURXqXu+Fe4sJ61SafKDJIGN9cqhinWOf2ZheKlM9VQ3UZyE16DW9axBqpFpj397lk0JmG9e86Eem1DWJNdZqR9bKchJRRllI3GMcNbY+x02B8N/Uh6QMCPCxbm0Y1nL5rfYYk2VSoSuOwzgsTx5IEoeDS2G9DDaJGYyY+llPfVNcURCjKvYc6QFr4aL7NbKCt7bOnG9ip0l0d34XyznPLN9tDyWFOGEXHEHkG96R9CosEWWyPjbA/hTonHR4dIn1CAd+a2B+lHDrZBTJG74YbTOgkjdpMlsDk6h2sFzBRvbSF9YkI1MKJpFLa+xTlIeSfYVGECbiJjUJT2yB1hrD84dpWGt4izko0VhDx1HsDJjFf2uJ2Bpd81csKWmBysHh5sKroGs2NOW0YhpdwM2PxBVsC4QrCHEXFzFs5lboEHebQ9IpJGwouAEjJ9CssWcutMWFCnBelwytelNDXFpCkJloRrMjct5H+1kDf0gdL8svP2E0fi/I6w1LIsjEaZo8TI7cTl750NaFIopm4NmqKjuatK5JzSwp05f+9NrW6fF/y/arFAlQt2S02SmhrXOg1/pftwVHipNAd4W73GEsAf2AZ8geQKI3hYJRJCjo2g+DxYYEuBBwC8zrLEs5BsI46bGmRwE5UmloZvQWefSBO5Mof71HStTE09tTmLYir04ceaxPicsawZYhvWB/C/gArya2QqpZMW5feo7HUHkE7jKZ+w2yMTzVOy3ml3wAgDWcIGd2eWwyCcksl2b8c6/3+/I2Z4BoK4cbXmVrnG/2CpMreRxKVkO7I+l3ll1EooD+WWczlMtgEI80tVdAHqbQj2NsB0jVegKry2AK6ra9SgCY3xBOgqvA948iggOyQJZBZ5CzSSvGUeX1fipJKpxP5RHo0SEb3ojXKQOnFKKWZ7+V3eidJp9Nfw++S25ok24aHX79oqFWB2WYhIFJoKWqpz4UYAnrwO2eabfZqbKKCaPca79G7BPhzoa1vS7a0qfMV5tVp79O+ByzPLftOU7SXBDt5SuM5q9S23+imZfFIerUhgK4zC/8qoGVWKVGT8hUsqcixlDZMn4wmMdAezwmQUKaQAkwyQCXgTQJdSSQiNJeADFxYyilR1xc+6Yr6Idr3HXEj9ijmNEmS3EpFOd9Yw6LN8RRY9ws6UqCyi+FNcAiiEg6FDZZa/LUroPgkQJBjyONIllEi9oqADC1GgeItcm1yqbCrnQhg0WPAXcPHWBN+Nk+pP8ixiPNSlML+vkjJ+Ryr3B2z9PP4NhZqGE3/CobjnLupaI14pt87D5SqcQKSgayEciIPKMioStlEBXxzyEg3soP05thQSyyJsb+uUI5k/HIk1VLLAI/qwwmMykkwe/rvDw89IQFPWlErJJBivXRsbjqHgAQgfmBFIw4AQI8q0EyV3mPHSOFdLlGNVKSV2XnCjQ/SbSoSzsY5AUNWfMjf4caSinjiP1PDGgbVp4NYyakY8hwLoKXzavKG9LifTtLQCJVsHzbNsFUd5FmjZwj7eG+8Ou2CE1PUALPaaLx877AVeCM0oY9chhp6k/0D+9duBIxgMefwZ3SUiZAXgcCBWX9D2a8RWnrNe29g76wg01LqkRcY2EqRQ9a6w/4U0iLWiEAEtCeqX94ZOQKHafhXhdabw5FSGrckh0o9tjijMgVhgmE1YIpT2xsKoOdjtoO7L5W2BvFXp6Ra/X/gEtHj0t4uhAFvs+S1h/Ax3m/Z6ZSxU//Y7OHCevfctk7RjQcWORdEY2Rh6I8vjssCGaGaMmibRXSENFN3ZLE4QuwYQ8r/63YsX9soYvThLEA0pA3XkAAFvbAXN7ZO027erUlXrQRmXhm3DaW5FB7cxyrButnJfOzIqY+5iFkz4x9KvqVOaMzPgURP6RHK9FQVcuG/UqErQ5VHkiMF3xMiSasrpwOx4qAd8tJPhHWpOTatoMZFC8k9zw3Dr/C/i96xD2fUT0xQLK2tnEGLF876MFEVAYCi/ua3rjAalDu2F+yLl2L0WwxZMB4/aQFqNXtM5d++2IRp6dbhFOkw1S8pi6hpMIBYbipU0h9CCbeY9jUfq2w4kUvU+iQFquKP3Yq+np1Y/dUeLEbZF7HWsT9d6NNmn3RIavZm8R7BESLEY+YUsVQvvAlFrP+Fl+9qxMGq7zTd2Wbf5xiyWputO5jsdQGm9Ufun4wUFiTqaeRVys7l94cMAabpBtXPVA7TVV1pniEPc25UpjWJ3nEcirOqpu3EHblUuRt+5dVBToBHB0HWVg3WI11Dqz+2/ZyJCMoPt9UMnRIujaFEHWdh9F1GBSfVFausTwx00XQQVm7UmzwoP9YaG/hUuOAfiDilS7nGS4XBPKOHTwPXYzow12c7/x2n68OjzFkzd316u1WL1oslrKRoqAYtzERhV3kwPPpWXW9zZ1BWXkaAVqPShvpjk2RvQ71vvMwkqtia/3ugYKeCueffdc5AU+U2Kmtu7nyY5gipRNJVDJ4Xme2tnNYd91t4igfeM2gCNojH/WooDKiSnU5z2C4lsnSZGjLmFASte8Yya0FEiPqK6cxns758zuP3IYUfk/fN6FXP8ZyDo2CONrk/ITHtZ5hkTqmJNVzXKWlMQfxzOKpUNoUPDsITrM6R9S6y+PlW/U/asZ+IZOi/MR3VVrHcadHtaXiXmJZLiPrFDBds5j6RwqsUEkPv6hDsjOpdMCpaMPvMWYXPbqck2weoJ+42qN/JpZHnTVNeBzfwJr30i7Up/pUFYaO9DOz0cB996qqn4kr0v62JxrbNa82A500Yor51U1bFEvwgaTXsK10x60WNmU1t6Eouv6VUNDjd+awCuSpThLKb4zFXlBXE8JEcVFvN4gXqAwrqVKaeuJLO+tm7ayFvuhHpbrIZGsUEnTGcE2iwWFUhUUZ5Jl5Hp081Uetlgkd5CLRw9zFu4F+0b/KpcWiISXD15I/y2Bm5dKCdC2lLMrcgyL4RslHQ2nPk6YTZBraRZXqifRXJQVVI+GtlQkazAkPKmvSq6lI+e/4wIOpHz+LwYuYa0xX0yaLj2dZWnubObed9GLgCdrOpZXmVaL/KwXiXLtfQbxZ9qGab8caaS1uJlK0Il+KdSfa34jeaIdrIdsT+3Q0remR1kqkMxjvbEkGxHrtg1Bs+2/IemaBjhHc9yA0x5sYG0S1CLkAQwNIpxl9sTsA0mgsqZlHLpBtQxPb2fdWCmtd3rkWQM5KB5R90zeg5V/CRQl4oOyL5o6BgV9+jXld36UiUbFDnP1Jf2JAwW9CauWqIxlAiGLWk5aDYY3qrhyz0RFpS3TjEPEVN1KZ067FA1AYpt8F59gWMMveapj3CM8c4fx0XG3QaEPk2PRDpfQJ2JT7oE5i3vIFv03/x327lXun6sC83bPhCdiyQalOY1TR5YhSEv2iozr8tu8schDh/Ya5+tCRAOabVPLJPdRG6nQU0rp7tvwxIyetu+bndvG8Ya+xm+x6Tf8WhM3Qgc49HvFA0TxtYFhj6aWZiVQurpqpVUoyfXUr39uObqGE8EmVQNAK8ukdaCrdqlZ+v35/ANEmpDo6uH0hhaRbV6Uayzg8O27kAvu8GEFJMHyMOjKQ7CBZ/8fx3K76lDeWYkhBtYBtOpIrpVnOGirKDABVvcBKwoXNT6EhS/3S8rn3ARXwgxbEpcUMPN/8YVf7dBo6ztChNy3+9l1K/gX4waO0eqTladYFA+2PoJP3Tpli1ddptTwh53CRACcJvFHbX9pkulbxp2iLHPh41wpD90vHovD+2QpHzJN+zQDEr6LS+bn1RTLemzKREza+cJpeqX0Qz1aZTfp/ybqbOGquEUNMG79b2tWR+bwVbpj49Xn/DJIE8659ySoWBhTFbCJ8OwMXsKe4yHbKzv/Iyp7gfONAcFlpTqT4IQdRHxgZBefmEtLQ/2h9xpCmC4iTENSptRIPPzPToezqtIzCuR+CgY9CeFeNXnwcan4BUPEMgwLzVSLl8DHObLijbCGX+DwL/4ihKlNYJgnkX41I3RcxzO5zQM7+K5oxFU50gG/ob860AHPHxNN25DhOsPjO/D9OJRJ/NIGHK9uPgHXHqxcSQjHQZ+Gpv+HMwTRIovuhmHdz9u44swvYjNbdyDTn+ZpX9J6k3ftyDyCy1PoJIqohcRfajlCTQkqk+hoS+29KJRynukT3ATZW9XoYLb4OaV2Xxs7iK5sSj0JrZU7SQRgL5nJ6Fr9WA6x43slJ2aOqCu4/p5aHqohpkq4Oy2GtY2SgWs3TbsihlzJaYwt2zWuot9qIgOum1oHBPG6SInJptMIH3OKBBqGDYOGwuAa9HthdOFAdz3eTTDdRXA0r0z1CrYAFn5oyqc5IY+IxbQIYVPOFI2PQhIioJAJriESA3+F5H6FH1tUwAA", "sha256": "2e9b9f47f51bc3571366883311f1f9200f839335c06aaa7c6e16bf7c602c24de"}}')
SNAPSHOT_ROOT.mkdir(parents=True, exist_ok=True)
for name, item in embedded.items():
    raw = gzip.decompress(base64.b64decode(item["payload"]))
    if hashlib.sha256(raw).hexdigest() != item["sha256"]:
        raise ValueError(f"Embedded file hash mismatch: {name}")
    target = SNAPSHOT_ROOT / name
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() != item["sha256"]:
        raise ValueError(f"Existing frozen source snapshot differs: {target}")
    target.write_bytes(raw)

if str(SNAPSHOT_ROOT) not in sys.path:
    sys.path.insert(0, str(SNAPSHOT_ROOT))
for module_name in (
    "cfpb_v052_pipeline_smoke_v02_common",
    "prepare_cfpb_seed_v052_pipeline_smoke_v02",
    "validate_cfpb_v052_pipeline_smoke_v02",
    "run_cfpb_v052_blind_semantic_judge_v01",
    "evaluate_cfpb_v052_semantic_judge_v01",
    "freeze_cfpb_v052_semantic_judge_v01",
):
    sys.modules.pop(module_name, None)
from cfpb_v052_pipeline_smoke_v02_common import sha256_file, load_semantic_judgments
from run_cfpb_v052_blind_semantic_judge_v01 import (
    RunnerConfig,
    fetch_zdr_endpoint_preflight,
    load_blind_cases,
    run_judge,
)
from validate_cfpb_v052_pipeline_smoke_v02 import validate_and_route
from evaluate_cfpb_v052_semantic_judge_v01 import evaluate_judge
from freeze_cfpb_v052_semantic_judge_v01 import freeze_judge

PROMPT = SNAPSHOT_ROOT / "cfpb_v052_semantic_judge_v01.md"
RUNNER_SOURCE = SNAPSHOT_ROOT / "run_cfpb_v052_blind_semantic_judge_v01.py"
COMMON_SOURCE = SNAPSHOT_ROOT / "cfpb_v052_pipeline_smoke_v02_common.py"
VALIDATOR_SOURCE = SNAPSHOT_ROOT / "validate_cfpb_v052_pipeline_smoke_v02.py"
print({name: item["sha256"] for name, item in embedded.items()})


## 3. Inspect payload, verify the live ZDR endpoint, and provide the API key


In [ ]:
from getpass import getpass
import pandas as pd

cases = load_blind_cases(RAW, PREPARED)
assert len(cases) == 20
assert all(set(case.model_dump()) == {"seed_id", "labels", "grounding", "generated"} for case in cases)
print({
    "rows": len(cases),
    "fields_visible_to_judge": ["case_id", "labels", "grounding_excerpt", "generated_dialogue"],
    "oracle_loaded_by_runner": False,
    "candidate_judgments_loaded_by_runner": False,
    "prompt_sha256": sha256_file(PROMPT),
})
display(pd.DataFrame([
    {
        "seed_id": case.seed_id,
        "product": case.labels["product"],
        "issue": case.labels["issue"],
        "grounding_preview": case.grounding[:180],
    }
    for case in cases[:2]
]))

CONFIG = RunnerConfig(
    model_name="qwen/qwen3.5-35b-a3b",
    base_url="https://openrouter.ai/api/v1",
    judge_id="openrouter_qwen3_5_35b_a3b_deepinfra_zdr_nonthinking_blind_v01_attempt03",
    sampling_profile="qwen3.5_official_instruct_general",
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    min_p=0.0,
    presence_penalty=1.5,
    repetition_penalty=1.0,
    max_tokens=2048,
    provider_slug="deepinfra",
    allow_fallbacks=False,
    require_parameters=True,
    data_collection="deny",
    zdr=True,
    reasoning_enabled=False,
    reasoning_exclude=True,
    seed=20260722,
    max_attempts=1,
    request_timeout_seconds=120.0,
)
print(CONFIG.model_dump())

endpoint_preflight = fetch_zdr_endpoint_preflight(
    config=CONFIG,
    output_path=ENDPOINT_PREFLIGHT,
)
print({
    "zdr_endpoint_preflight": "passed",
    "provider": endpoint_preflight.selected_endpoint.provider_name,
    "tag": endpoint_preflight.selected_endpoint.tag,
    "status": endpoint_preflight.selected_endpoint.status,
    "supported_parameters": endpoint_preflight.selected_endpoint.supported_parameters,
})

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OPENROUTER_API_KEY (not persisted): ")
if not os.environ["OPENROUTER_API_KEY"].strip():
    raise RuntimeError("OPENROUTER_API_KEY is empty")


## 4. Two-row API/schema contract smoke

These fixed development IDs exercise one previously accepted row and
one semantic-reject row. Their expected labels are not sent or loaded.
Rerunning this cell uses the separate two-row cache.


In [ ]:
CONTRACT_IDS = [
    "cfpb_v052_0d210ece132c66d3",
    "cfpb_v052_2c70d920e4cfccd5",
]
contract_report = run_judge(
    raw_path=RAW,
    prepared_path=PREPARED,
    prompt_path=PROMPT,
    judgments_path=CONTRACT_ROOT / "semantic_judgments.jsonl",
    raw_responses_path=CONTRACT_ROOT / "raw_responses.jsonl",
    failed_attempts_path=CONTRACT_ROOT / "failed_attempts.jsonl",
    endpoint_preflight_path=ENDPOINT_PREFLIGHT,
    report_path=CONTRACT_ROOT / "judge_run_report.json",
    config=CONFIG,
    api_key=os.environ["OPENROUTER_API_KEY"],
    selected_seed_ids=CONTRACT_IDS,
)
assert contract_report["semantic_coverage_complete"] is True
contract = load_semantic_judgments(CONTRACT_ROOT / "semantic_judgments.jsonl")
display(pd.DataFrame([
    {"seed_id": key, "decision": value.decision, "reasons": value.reasons}
    for key, value in sorted(contract.items())
]))


## 5. Run/resume the blind 20-row development calibration


In [ ]:
import shutil

# Reuse the two contract-smoke judgments only when starting a fresh
# calibration. They used the exact same prompt, config, and schema;
# their separate originals remain as contract evidence.
calibration_judgments = CALIBRATION_ROOT / "semantic_judgments.jsonl"
calibration_responses = CALIBRATION_ROOT / "raw_responses.jsonl"
calibration_failures = CALIBRATION_ROOT / "failed_attempts.jsonl"
if not calibration_judgments.exists() and not calibration_responses.exists():
    CALIBRATION_ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CONTRACT_ROOT / "semantic_judgments.jsonl", calibration_judgments)
    shutil.copy2(CONTRACT_ROOT / "raw_responses.jsonl", calibration_responses)
    contract_failures = CONTRACT_ROOT / "failed_attempts.jsonl"
    if contract_failures.exists():
        shutil.copy2(contract_failures, calibration_failures)
    print("Bootstrapped the fresh calibration cache with 2 contract-smoke rows")
elif calibration_judgments.exists() != calibration_responses.exists():
    raise ValueError("Calibration judgment/audit cache is incomplete")

calibration_report = run_judge(
    raw_path=RAW,
    prepared_path=PREPARED,
    prompt_path=PROMPT,
    judgments_path=calibration_judgments,
    raw_responses_path=calibration_responses,
    failed_attempts_path=calibration_failures,
    endpoint_preflight_path=ENDPOINT_PREFLIGHT,
    report_path=CALIBRATION_ROOT / "judge_run_report.json",
    config=CONFIG,
    api_key=os.environ["OPENROUTER_API_KEY"],
)
assert calibration_report["semantic_coverage_complete"] is True
print({
    "rows": calibration_report["judgment_rows"],
    "new_provider_calls": calibration_report["new_provider_calls"],
    "decision_counts": calibration_report["decision_counts"],
    "failed_attempt_rows": calibration_report["outputs"]["failed_attempts"]["rows"],
})


## 6. Score only after all blind judgments exist

This is the first cell that loads the GPT-5.6-SOL development
adjudication. It measures agreement, not human accuracy.


In [ ]:
EVALUATED = CALIBRATION_ROOT / "evaluated"
evaluation = validate_and_route(
    raw_output=RAW,
    prepared_input=PREPARED,
    input_manifest=INPUT_MANIFEST,
    disposition_path=PRIVACY_DISPOSITION,
    validated_path=EVALUATED / "validated/dialogues.jsonl",
    rejected_path=EVALUATED / "rejected/dialogues.jsonl",
    review_path=EVALUATED / "review/dialogues.jsonl",
    report_path=EVALUATED / "validation_report_v02.json",
    semantic_judgments_path=CALIBRATION_ROOT / "semantic_judgments.jsonl",
    oracle_path=ORACLE,
)
metrics = evaluation["oracle_metrics"]
calibration_metrics = evaluate_judge(
    judgments_path=CALIBRATION_ROOT / "semantic_judgments.jsonl",
    oracle_path=ORACLE,
    combined_report_path=EVALUATED / "validation_report_v02.json",
    output_path=CALIBRATION_ROOT / "calibration_metrics_v01.json",
)
display(pd.DataFrame({
    "judge_only": pd.Series(calibration_metrics["judge_only_metrics"]),
    "combined_validator": pd.Series(calibration_metrics["combined_validator_metrics"]),
}))
print({
    "validated": evaluation["validated_rows"],
    "rejected": evaluation["rejected_rows"],
    "review": evaluation["review_rows"],
    "development_only": True,
    "human_gold": False,
})


## 7. Explicitly freeze judge v01

Review the metrics above first. Freezing records the exact model,
prompt, parameters, schemas, runner, source inputs, judgments, and
calibration report. It does not approve a benchmark or formal pilot.


In [ ]:
CALIBRATION_REVIEW_COMPLETED = False
APPROVE_FREEZE = False
APPROVER_ID = ""

if not CALIBRATION_REVIEW_COMPLETED or not APPROVE_FREEZE:
    print(
        "Not frozen. Non-thinking execution success alone is not freeze evidence. "
        "Review the 20-row metrics and disagreements first; only then set both "
        "CALIBRATION_REVIEW_COMPLETED=True and APPROVE_FREEZE=True with APPROVER_ID."
    )
else:
    frozen = freeze_judge(
        judge_report_path=CALIBRATION_ROOT / "judge_run_report.json",
        judgments_path=CALIBRATION_ROOT / "semantic_judgments.jsonl",
        evaluation_path=EVALUATED / "validation_report_v02.json",
        metrics_path=CALIBRATION_ROOT / "calibration_metrics_v01.json",
        prompt_path=PROMPT,
        oracle_path=ORACLE,
        raw_path=RAW,
        prepared_path=PREPARED,
        runner_path=RUNNER_SOURCE,
        common_path=COMMON_SOURCE,
        validator_path=VALIDATOR_SOURCE,
        endpoint_preflight_path=ENDPOINT_PREFLIGHT,
        output_path=REVALIDATION_ROOT / "judges/frozen/cfpb_v052_semantic_judge_v01_attempt03.json",
        approved_by=APPROVER_ID,
        approve_freeze=True,
    )
    print(json.dumps(frozen.model_dump(mode="json"), ensure_ascii=False, indent=2))


## Stop point

Do not revise the frozen prompt, reason definitions, model, inference
parameters, or parsing logic after this notebook. The next notebook
must generate a new held-out smoke and apply this frozen judge without
calibrating on the held-out answers. A finite human audit follows that
fixed evaluation. Formal 50–100-row generation remains blocked until a
separately approved privacy protocol makes
`benchmark_release_gate_passed=true`.
